# Financer Name Entity Resolution Pipeline

Cluster similar `rc_financer` strings into a single canonical financer while **minimizing false positives**.

| Item | Value |
|------|-------|
| Dataset | `total_financers.csv` (~738,115 unique raw strings) |
| Embedding model | Qwen Embedding (`Qwen/Qwen3-Embedding-*`) |
| Search engine | FAISS (`IndexFlatIP`) |
| Clustering | Union-Find (Disjoint Set) |

**Pipeline**: clean -> typo/OCR fix -> abbreviation expansion -> brand alias fold -> institution
resolution (branch collapse) -> canonical name -> embed -> FAISS top-K -> features -> rule engine
-> union-find -> canonical financer.

---

## How this differs from the owner pipeline

The owner pipeline is the template. Four things are genuinely different in the financer domain and
each one gets its own phase or keyword set here:

1. **Branch suffixes are noise, not identity.** `STATE BANK OF INDIA HATHUR`,
   `STATE BANK OF INDIA,UDUPI BRANCH` and `SBI, PB BR DHANBAD` are all *State Bank of India*.
   Phase 3 resolves the institution against a gazetteer and pushes the residue into `branch_hint`.
   (In the owner pipeline the analogous tokens *were* identity — `A J IMPEX` != `A J BUILDERS`.)
2. **Abbreviation ladders are everywhere.** `FIN / FINA / FINAN / FINANC / FINACE / FINANACE`,
   `INV / INVE / INVES / INVEST / INVST / INVT`, `SER / SERV`, `CR / CRE`. Phase 2.2 expands them to
   one surface form before anything else compares strings.
3. **Co-operative-society vocabulary is transliterated many ways.**
   `SAH / SAHAKARI / SAHKARI / SAHAKARA`, `SOUHARDA / SOUHARD / SOUHARDHA`,
   `PAT / PATH / PATTINA / PATSANSTHA / PATHASANTHA`, `GRAMIN / GRAMEEN / GRAMEENA`,
   `NAGARI / NAGRI / NAGARIK`, `MARYADIT / MARYA` (= "limited", so it is a *legal form*, not a descriptor).
4. **`STATE`, `CENTRAL`, `NATIONAL`, `UNION`, `INDIA` are discriminative here.** In the owner pipeline
   `INDIA` is a throwaway descriptor. Here it separates `STATE BANK OF INDIA` from `BANK OF INDIA`
   from `CENTRAL BANK OF INDIA` from `UNION BANK OF INDIA`. They are deliberately **kept out** of
   `DESCRIPTOR_WORDS` so they stay in `core_tokens`.

## Step 0 : Setup & Configuration

In [ ]:
# Step 0.1 : Install dependencies (run once, uncomment)
# !pip install pandas numpy faiss-cpu sentence-transformers rapidfuzz python-Levenshtein tqdm unidecode pyarrow
# For GPU embedding + search use: faiss-gpu and a CUDA-enabled torch

In [ ]:
# Step 0.2 : Imports
import os
import re
import json
import unicodedata
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Fuzzy string metrics
from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein

# Embedding + search (imported lazily in their phases to keep this cell light)
# from sentence_transformers import SentenceTransformer
# import faiss

tqdm.pandas()

In [ ]:
# Step 0.3 : Global config / paths
CONFIG = {
    # ---- IO ----
    "input_file":        "total_financers.csv",   # single column, header = rc_financer
    "input_col":         "rc_financer",
    "artifacts_dir":     "artifacts",

    # ---- Embedding ----
    # 738k rows x 4096 dim float32 = ~12 GB. Phase 5 embeds only the DISTINCT canonical names
    # (institution collapse cuts the distinct count hard), then maps vectors back to rows.
    # Drop to "Qwen/Qwen3-Embedding-0.6B" (1024-dim) if VRAM/RAM is tight.
    "model_name":        "Qwen/Qwen3-Embedding-8B",
    "batch_size":        256,
    "normalize":         True,          # L2-normalize embeddings so IP == cosine

    # ---- FAISS ----
    "top_k":             100,           # neighbors per name

    # ---- Rule engine thresholds (tune in Phase 9) ----
    "rules": {
        "r1_cosine":     0.97, "r1_token_set": 95, "r1_rapidfuzz": 90,
        "r2_lev":        2,    "r2_cosine":    0.94,
        "min_cosine_gate": 0.85,   # coarse pruning gate before feature calc
        "core_sim_min":  85,       # distinguishing-part similarity floor
        "near_exact_rf": 95, "near_exact_cos": 0.97,   # full-string near-identity bypass
        "r3_cosine":     0.95, "r3_core_sim": 90,      # abbreviation-expansion merge
        # financer-specific
        "r4_inst_cos":   0.90,     # both sides resolved to the SAME gazetteer institution ->
                                   # a low bar is enough, the gazetteer already proved identity
        # ---- anti-chaining guards (Phase 9) ----
        "r3_cosine_unanchored": 0.97,   # R3 is the chaining rule; unanchored names get a higher bar
        "core_jaccard_min":     0.5,    # both sides multi-core -> cores must overlap this much
        "core_extra_max":       2,      # subset match with >= this many extra core tokens -> reject
    },

    # ---- Clustering (Phase 10) ----
    "cluster": {
        # Union-Find is a transitive closure: A~B, B~C merges A and C even though nothing ever
        # compared them. Once a component holds this many distinct names, every further edge must
        # ALSO pass decide() between the two component REPRESENTATIVES.
        "rep_check_min_size": 10,
    },

    # ---- Institution resolution (Phase 3) ----
    "branch": {
        # a gazetteer hit is only trusted when it starts at token 0 (after THE / M S are dropped)
        "anchor_at_start_only": True,
        # residue kept for audit; never used for clustering
        "keep_branch_hint": True,
    },
}

os.makedirs(CONFIG["artifacts_dir"], exist_ok=True)

def art(name):
    '''Path helper inside artifacts dir.'''
    return os.path.join(CONFIG["artifacts_dir"], name)

CONFIG

## Phase 1 : Data Cleaning
Standardize names before anything else. Uppercase, collapse spaces, replace punctuation,
unicode-normalize, trim. Cleaning must **never remove meaningful words**.

Financer strings carry more punctuation garbage than owner strings
(`CHOLAMANDALAM INVE.&FIN.COM. LTD.`, `C.I,F.C.L`, `SBI,  PB BR DHANBAD`), so the
punctuation-to-space rule does most of the heavy lifting here.

In [ ]:
# Step 1.1 : Load raw data
df = pd.read_csv(CONFIG["input_file"])
df = df.rename(columns={CONFIG["input_col"]: "original_name"})
df["original_name"] = df["original_name"].astype(str)

# raw file is already de-duplicated, but never assume it
before = len(df)
df = df.drop_duplicates("original_name").reset_index(drop=True)
print("rows:", len(df), f"(dropped {before - len(df)} exact duplicates)")
df.head(10)

In [ ]:
# Step 1.2 : Cleaning function
# & -> AND ; other punctuation ( , . - / ( ) etc ) -> space ; unicode accents stripped
_PUNCT_TO_SPACE = re.compile(r"[^A-Z0-9&\s]")
_MULTISPACE     = re.compile(r"\s+")

def clean_name(name: str) -> str:
    if not isinstance(name, str):
        return ""
    # unicode normalize: E' -> E (strip accents)
    s = unicodedata.normalize("NFKD", name)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    # uppercase
    s = s.upper()
    # ampersand -> AND (spaced so it becomes its own token)
    s = s.replace("&", " AND ")
    # remaining punctuation -> space (keeps digits & letters, drops , . - / ( ) etc)
    s = _PUNCT_TO_SPACE.sub(" ", s)
    # collapse multi-space + trim
    s = _MULTISPACE.sub(" ", s).strip()
    return s

In [ ]:
# Step 1.3 : Apply cleaning
df["clean_name"] = df["original_name"].map(clean_name)
df[["original_name", "clean_name"]].head(10)

In [ ]:
# Step 1.4 : Validation - inspect 100 random records; confirm no meaningful words dropped
sample = df.sample(100, random_state=42)[["original_name", "clean_name"]]
for o, c in sample.itertuples(index=False):
    print(f"{o!r:60} -> {c!r}")

## Phase 2 : Typo / OCR Normalization + Abbreviation Expansion
Two token-level dictionaries, applied in order:

1. **`OCR_DICT`** - misspellings and scanner errors of *financer* words
   (`FINANACE`->`FINANCE`, `C0` -> `CO`, `FTRST`->`FIRST`, `MAHARASHARA`->`MAHARASHTRA`).
2. **`ABBREV_DICT`** - deliberate truncations, expanded to one surface form
   (`FIN`/`FINA`/`FINAN`->`FINANCE`, `INV`/`INVE`/`INVST`->`INVESTMENT`, `SAH`->`SAHAKARI`).

Both are **exact-token** replacements, so person / village / district names are never touched.

> **Known trade-off**: `CR` -> `CREDIT` and `SER` -> `SERVICES` also fire when those tokens are
> somebody's initials. In this dataset the abbreviation reading dominates by orders of magnitude
> (`CR` 9,464 occurrences, almost all `CREDIT`). Move a token into `PROTECTED_TOKENS` to opt it out.

In [ ]:
# Step 2.1 : OCR / misspelling dictionary (financer words only)
# Keys are wrong tokens, values are correct tokens. Extend as new OCR errors are found.
OCR_DICT = {
    # ---- legal form ----
    "LIMITD": "LIMITED", "LIMTED": "LIMITED", "LIMITEDD": "LIMITED", "LIMITE": "LIMITED",
    "LIMIED": "LIMITED", "LMITED": "LIMITED", "LIMIITED": "LIMITED", "LIMTIED": "LIMITED",
    "LIITED": "LIMITED", "LIMIETD": "LIMITED", "LIMITES": "LIMITED", "LTMITED": "LIMITED",
    "LINITED": "LIMITED", "LITIMED": "LIMITED", "IMITED": "LIMITED", "LIMATED": "LIMITED",
    "LIIMTED": "LIMITED", "LITED": "LIMITED", "LIMITID": "LIMITED", "LEMITED": "LIMITED",
    "LOMITED": "LIMITED", "TIMITED": "LIMITED", "LIMUTED": "LIMITED", "LITIED": "LIMITED",
    "LEMITE": "LIMITED",
    "TLD": "LIMITED", "LDT": "LIMITED", "LID": "LIMITED", "LTF": "LIMITED", "LTS": "LIMITED",
    "LTDF": "LIMITED", "LTDQ": "LIMITED", "LITD": "LIMITED", "TLTD": "LIMITED",
    "LLTD": "LIMITED", "LTT": "LIMITED", "LTE": "LIMITED", "LYD": "LIMITED", "LTA": "LIMITED",
    "LDH": "LIMITED", "LLT": "LIMITED", "PTD": "LIMITED",
    "PRIVATELIMITED": "PRIVATE LIMITED", "PVTLTD": "PRIVATE LIMITED",
    "PRIVAT": "PRIVATE", "PRAVATE": "PRIVATE", "PRIVATED": "PRIVATE", "PRIVTE": "PRIVATE",
    "PRVATE": "PRIVATE", "PRIVETE": "PRIVATE", "PRIVITE": "PRIVATE",
    "COMPNY": "COMPANY", "COMPANYY": "COMPANY", "COMAPANY": "COMPANY",
    "CORPORATON": "CORPORATION", "CORPORTION": "CORPORATION", "CORPARATION": "CORPORATION",

    # ---- finance vocabulary ----
    "FINANACE": "FINANCE", "FINACE": "FINANCE", "FINANCEE": "FINANCE", "FINANC": "FINANCE",
    "FINACNE": "FINANCE", "FINANCS": "FINANCE", "FIANANCE": "FINANCE", "FINNACE": "FINANCE",
    "FINANEC": "FINANCE", "FIINANCE": "FINANCE", "FIIN": "FIN", "FINANCIALS": "FINANCIAL",
    "FINANCIAI": "FINANCIAL", "FINACIAL": "FINANCIAL", "FINANCIL": "FINANCIAL",
    "FINACIALS": "FINANCIAL", "FINANCEIAL": "FINANCIAL",
    "FINSERVE": "FINSERV", "FINSERVICES": "FINSERV", "FINSREV": "FINSERV",
    "INVESTMANT": "INVESTMENT", "INVESMENT": "INVESTMENT", "INVSTMENT": "INVESTMENT",
    "INVESTEMENT": "INVESTMENT", "INVESTMEN": "INVESTMENT",
    "CREDITT": "CREDIT", "CRIDIT": "CREDIT", "CREDT": "CREDIT", "CERDIT": "CREDIT",
    "LEASNG": "LEASING", "LESING": "LEASING", "LEASHING": "LEASING",
    "CAPITALS": "CAPITAL", "CAPTAL": "CAPITAL", "CAPITL": "CAPITAL", "CAPTIAL": "CAPITAL",
    "SECURITES": "SECURITIES", "SECURITITES": "SECURITIES",
    "HOLDING": "HOLDINGS", "HOLDIGS": "HOLDINGS",

    # ---- bank / society vocabulary ----
    "BNAK": "BANK", "BAKN": "BANK", "BANL": "BANK", "BABK": "BANK", "BANNK": "BANK",
    "BAN": "BANK", "BNK": "BANK",
    "BANKLTD": "BANK LIMITED", "BANKLIMITED": "BANK LIMITED",
    "COOPERATIV": "COOPERATIVE", "CO0PERATIVE": "COOPERATIVE", "COOPRATIVE": "COOPERATIVE",
    "COOPERTIVE": "COOPERATIVE", "COPERATIVE": "COOPERATIVE", "CORPERATIVE": "COOPERATIVE",
    "C0": "CO", "C0OP": "COOP", "CO0P": "COOP",
    "SOCIETI": "SOCIETY", "SOCIET": "SOCIETY", "SOCEITY": "SOCIETY", "SOCIETYY": "SOCIETY",
    "SOCIE": "SOCIETY", "SOSIETY": "SOCIETY",
    "SANSTHAN": "SANSTHA", "SANSTA": "SANSTHA", "SANTHA": "SANSTHA", "SASNTHA": "SANSTHA",

    # ---- brand-name misspellings (folded to the correct brand spelling) ----
    "CHOLAMANDLAM": "CHOLAMANDALAM", "CHOLAMANDALM": "CHOLAMANDALAM",
    "CHOLAMANDLAM": "CHOLAMANDALAM", "CHOLAMANDALAMM": "CHOLAMANDALAM",
    "CHOLAMANDLAMM": "CHOLAMANDALAM", "CHOLMANDALAM": "CHOLAMANDALAM",
    "SHRRAM": "SHRIRAM", "SHRIRAAM": "SHRIRAM", "SRIRAM": "SHRIRAM", "SHREERAM": "SHRIRAM",
    "MAHINDRAA": "MAHINDRA", "MAHENDRA": "MAHINDRA", "MAHINDA": "MAHINDRA",
    "FTRST": "FIRST", "FRIST": "FIRST",
    "MAHARASHARA": "MAHARASHTRA", "MAHARASTRA": "MAHARASHTRA", "MAHARSHTRA": "MAHARASHTRA",
    "KARNATAKA": "KARNATAKA", "KARNATKA": "KARNATAKA", "KARANATAKA": "KARNATAKA",
    "CTY": "CITY", "CITI": "CITY",
    "LICI": "LIC",
    "INIDA": "INDIA", "INDAI": "INDIA", "IDNIA": "INDIA", "INDIYA": "INDIA",
    "INDA": "INDIA", "INDIAN": "INDIAN",
}

# Tokens that must NEVER be rewritten, even if a dictionary above lists them.
# Put a token here when its abbreviation reading is wrong more often than it is right.
PROTECTED_TOKENS = set()

print("ocr entries:", len(OCR_DICT))

In [ ]:
# Step 2.2 : Abbreviation dictionary (truncations -> one surface form)
#
# The single biggest source of false NEGATIVES in this dataset. Without this,
# "CHOLAMANDALAM INV FIN COM LTD" and "CHOLAMANDALAM INVESTMENT FINANCE COMPANY LIMITED"
# share almost no tokens and never merge.
ABBREV_DICT = {
    # finance
    "FIN": "FINANCE", "FINA": "FINANCE", "FINAN": "FINANCE", "FINANCES": "FINANCE",
    "FINC": "FINANCE", "FNC": "FINANCE", "FNCE": "FINANCE",
    "FINL": "FINANCIAL", "FINAL": "FINANCIAL", "FNL": "FINANCIAL",
    "FINANCIERS": "FINANCIERS", "FINANCIER": "FINANCIERS",
    "INV": "INVESTMENT", "INVE": "INVESTMENT", "INVES": "INVESTMENT", "INVEST": "INVESTMENT",
    "INVST": "INVESTMENT", "INVT": "INVESTMENT", "INVTS": "INVESTMENT",
    "INVESTMENTS": "INVESTMENT", "INVESTMET": "INVESTMENT",
    "CAP": "CAPITAL",
    "CR": "CREDIT", "CRE": "CREDIT", "CRED": "CREDIT", "CREDITS": "CREDIT", "CRD": "CREDIT",
    "LEAS": "LEASING", "LSG": "LEASING",
    "HPA": "HIRE PURCHASE",   # hire-purchase agreement
    "MTR": "MOTORS", "MOTOR": "MOTORS",
    # NOT expanded on purpose:
    #   HP  -> "HIRE PURCHASE"  collides with Himachal Pradesh ("THE HP STATE CO OP BANK")
    #   SEC -> "SECURITIES"     collides with "SEC 5" / sector in branch addresses
    #   ST  -> "STATE"          collides with Saint ("ST XAVIER")
    #   MAH -> "MAHILA"         collides with Maharashtra ("MAH GRAMIN BANK")

    # generic company words
    "COM": "COMPANY", "COMP": "COMPANY", "COY": "COMPANY", "CMP": "COMPANY",
    "SER": "SERVICES", "SERV": "SERVICES", "SERVICE": "SERVICES", "SVC": "SERVICES",
    "SERVS": "SERVICES", "SRV": "SERVICES",
    "CORP": "CORPORATION", "CORPN": "CORPORATION",
    "ENT": "ENTERPRISES", "ENTP": "ENTERPRISES", "ENTERPRISE": "ENTERPRISES",
    "IND": "INDIA", "INDIAA": "INDIA",

    # bank / branch structure
    "BR": "BRANCH", "BRH": "BRANCH", "BRN": "BRANCH", "BRANC": "BRANCH", "BRCH": "BRANCH",
    "BRANCHES": "BRANCH", "BRAN": "BRANCH", "BRA": "BRANCH",
    "DIST": "DISTRICT", "DISTT": "DISTRICT", "DIS": "DISTRICT", "DT": "DISTRICT",
    "CEN": "CENTRAL", "CENT": "CENTRAL", "CNTRL": "CENTRAL",
    "NAT": "NATIONAL", "NATL": "NATIONAL",
    "UP": "UTTAR PRADESH",     # "UP GRAMIN BANK" -> "UTTAR PRADESH GRAMIN BANK"

    # co-operative / society vocabulary (Marathi / Kannada / Hindi transliterations)
    "SAH": "SAHAKARI", "SAHA": "SAHAKARI", "SAHK": "SAHAKARI", "SAHKARI": "SAHAKARI",
    "SAHAKARA": "SAHAKARI", "SAHAKAR": "SAHAKARI", "SAHAKARY": "SAHAKARI",
    "SOU": "SOUHARDA", "SOUHARD": "SOUHARDA", "SOUHARDHA": "SOUHARDA",
    "SOUHARDHA": "SOUHARDA", "SAUHARDA": "SOUHARDA", "SOUHARDA": "SOUHARDA",
    "PAT": "PATSANSTHA", "PATH": "PATSANSTHA", "PATT": "PATSANSTHA",
    "PATTIN": "PATSANSTHA", "PATTINA": "PATSANSTHA", "PATHSANSTHA": "PATSANSTHA",
    "PATHASANTHA": "PATSANSTHA", "PATSANSTA": "PATSANSTHA", "PATSANTHA": "PATSANSTHA",
    "PATHSANSTA": "PATSANSTHA", "PATSANSTHAN": "PATSANSTHA",
    "NAG": "NAGARI", "NAGRI": "NAGARI", "NAGARIK": "NAGARI", "NAGRIK": "NAGARI",
    "NAGARIKA": "NAGARI",
    "GRA": "GRAMIN", "GRAM": "GRAMIN", "GRAMEEN": "GRAMIN", "GRAMEENA": "GRAMIN",
    "GRAMINA": "GRAMIN", "GRAMEN": "GRAMIN",
    "BIG": "BIGARSHETI", "BIGAR": "BIGARSHETI", "BIGARSHETI": "BIGARSHETI",
    "SHE": "SHETI", "SHETKARI": "SHETI",
    "SAN": "SANSTHA", "SANS": "SANSTHA", "SANST": "SANSTHA",
    "SANGH": "SANGHA", "SANG": "SANGHA", "SANGHAM": "SANGHA",
    "SOC": "SOCIETY", "SOCY": "SOCIETY", "STY": "SOCIETY",
    "MHL": "MAHILA",
    "SA": "SAHAKARI", "MAR": "MARYADIT", "GR": "GRAMIN", "NI": "NIYAMIT",
    "JANTA": "JANATA", "JANATHA": "JANATA",
    "URB": "URBAN", "UR": "URBAN",
    "MULTI": "MULTIPURPOSE", "MULTIPUR": "MULTIPURPOSE",
    "VIVIDODESHA": "MULTIPURPOSE", "VIVIDODDESHAGALA": "MULTIPURPOSE",
    "VIVIDHODDESHA": "MULTIPURPOSE",
    "MARY": "MARYADIT", "MARYA": "MARYADIT", "MRYDT": "MARYADIT", "MYDT": "MARYADIT",
    "NIY": "NIYAMIT", "NIYA": "NIYAMIT", "NYT": "NIYAMIT", "NIYAMITHA": "NIYAMIT",
    "REGD": "REGISTERED", "REG": "REGISTERED",
    "CO OPERATIVE": "COOPERATIVE",   # handled by the bigram pass below, listed here for reference
    "OP": "COOPERATIVE", "COOP": "COOPERATIVE", "OPERATIVE": "COOPERATIVE",
    "CPRTV": "COOPERATIVE",
}

# Bigrams collapsed BEFORE the unigram pass ("CO OP" is two tokens after cleaning).
ABBREV_BIGRAMS = {
    ("CO", "OP"):          "COOPERATIVE",
    ("CO", "OPERATIVE"):   "COOPERATIVE",
    ("CO", "OPRATIVE"):    "COOPERATIVE",
    ("HIRE", "PURCHASE"):  "HIRE PURCHASE",
    ("BIGAR", "SHETI"):    "BIGARSHETI",
    ("BIG", "SHE"):        "BIGARSHETI",
    ("PAT", "SANSTHA"):    "PATSANSTHA",
    ("PATH", "SANSTHA"):   "PATSANSTHA",
}

_overlap = set(OCR_DICT) & set(ABBREV_DICT)
print("abbrev entries:", len(ABBREV_DICT), "| bigrams:", len(ABBREV_BIGRAMS),
      "| overlap with OCR_DICT:", sorted(_overlap))

In [ ]:
# Step 2.3 : Brand alias map (initialisms / short forms -> the brand's full token form)
#
# Pure rules cannot connect "CIFCL" to "CHOLAMANDALAM" or "MMFSL" to "MAHINDRA" - they share no
# characters. This map is the only place that knowledge lives. Keys are exact tokens.
BRAND_ALIASES = {
    # NBFC initialisms
    "CIFCL":  "CHOLAMANDALAM INVESTMENT FINANCE",
    "CIFC":   "CHOLAMANDALAM INVESTMENT FINANCE",
    "CHOLA":  "CHOLAMANDALAM",
    "MANDALAM": "CHOLAMANDALAM",         # "...INVE.&FIN" strings that split the brand
    "MANDLAM":  "CHOLAMANDALAM",
    "MMFSL":  "MAHINDRA FINANCIAL SERVICES",
    "MMFS":   "MAHINDRA FINANCIAL SERVICES",
    "MAHINDRAFINANCE": "MAHINDRA FINANCE",
    "SCUF":   "SHRIRAM CITY UNION FINANCE",
    "STFC":   "SHRIRAM TRANSPORT FINANCE",
    "SFL":    "SHRIRAM FINANCE",
    "BFL":    "BAJAJ FINANCE",
    "TMFL":   "TATA MOTORS FINANCE",
    "TCL":    "TATA CAPITAL",
    "HDB":    "HDB FINANCIAL SERVICES",
    "HDBFS":  "HDB FINANCIAL SERVICES",
    "SMFG":   "SMFG INDIA CREDIT",
    "FULLERTON": "SMFG INDIA CREDIT",    # Fullerton India -> renamed SMFG India Credit
    "IIFL":   "IIFL FINANCE",
    "MUTHOOT": "MUTHOOT",
    "TVSCS":  "TVS CREDIT SERVICES",
    "SUNDARAMFINANCE": "SUNDARAM FINANCE",
    "AUSFB":  "AU SMALL FINANCE BANK",
    "ESAF":   "ESAF SMALL FINANCE BANK",
    "CNHI":   "CNH INDUSTRIAL CAPITAL",
    "JDCC":   "JOHN DEERE FINANCIAL",

    # bank initialisms
    "SBI":    "STATE BANK OF INDIA",
    "SBH":    "STATE BANK OF HYDERABAD",
    "SBP":    "STATE BANK OF PATIALA",
    "SBT":    "STATE BANK OF TRAVANCORE",
    "SBM":    "STATE BANK OF MYSORE",
    "PNB":    "PUNJAB NATIONAL BANK",
    "BOB":    "BANK OF BARODA",
    "BOI":    "BANK OF INDIA",
    "BOM":    "BANK OF MAHARASHTRA",
    "CBI":    "CENTRAL BANK OF INDIA",
    "UBI":    "UNION BANK OF INDIA",
    "IOB":    "INDIAN OVERSEAS BANK",
    "UCO":    "UCO BANK",
    "IDBI":   "IDBI BANK",
    "IDFC":   "IDFC FIRST BANK",
    "HDFC":   "HDFC BANK",
    "ICICI":  "ICICI BANK",
    "AXIS":   "AXIS BANK",
    "KOTAK":  "KOTAK MAHINDRA BANK",
    "KMBL":   "KOTAK MAHINDRA BANK",
    "INDUSIND": "INDUSIND BANK",
    "IBL":    "INDUSIND BANK",
    "JK":     "JAMMU AND KASHMIR BANK",
    "JKB":    "JAMMU AND KASHMIR BANK",
    "APGVB":  "ANDHRA PRADESH GRAMEENA VIKAS BANK",
    "PSB":    "PUNJAB AND SIND BANK",
    "RRB":    "REGIONAL RURAL BANK",
    "LIC":    "LIC",
}

# Multi-token aliases: matched on the token sequence, replaced by the canonical string.
# Runs of bare initials ("S B I", "H D F C") are joined by Step 2.4 BEFORE this, so those do not
# need an entry here - only sequences held apart by a real word (AND) or a genuine word pair do.
BRAND_ALIAS_PHRASES = {
    ("J", "AND", "K"):            "JAMMU AND KASHMIR",
    ("M", "AND", "M"):            "MAHINDRA AND MAHINDRA",
    ("MAHI", "AND", "MAHI"):      "MAHINDRA AND MAHINDRA",
    ("MAHILA", "AND", "MAHILA"):  "MAHINDRA AND MAHINDRA",   # OCR of M&M; MAHILA alone is untouched
    ("SRI", "RAM", "FINANCE"):    "SHRIRAM FINANCE",
    ("SHRI", "RAM", "FINANCE"):   "SHRIRAM FINANCE",
    ("SHREE", "RAM", "FINANCE"):  "SHRIRAM FINANCE",
    ("SRI", "RAM", "CITY"):       "SHRIRAM CITY",
    ("SHRI", "RAM", "CITY"):      "SHRIRAM CITY",
    ("SRI", "RAM", "TRANSPORT"):  "SHRIRAM TRANSPORT",
    ("SHRI", "RAM", "TRANSPORT"): "SHRIRAM TRANSPORT",
    ("MAH", "AND", "MAH"):        "MAHINDRA AND MAHINDRA",
    ("HP", "STATE"):              "HIMACHAL PRADESH STATE",
    ("HP", "GRAMIN"):             "HIMACHAL PRADESH GRAMIN",
}

# Whole-token regex folds for brands whose misspellings are too numerous to enumerate.
# Anchored on both ends so they can only rewrite a complete token.
BRAND_REGEX = [
    (re.compile(r"^CHOLA?M[A-Z]*$|^CHOLA$"), "CHOLAMANDALAM"),   # CHOLAMADALAM, CHOLMANDALAM, ...
    (re.compile(r"^MAHIN(D|DR)[A-Z]*$"),     "MAHINDRA"),        # MAHINDA, MAHINDRAA, ...
    (re.compile(r"^SH?R?EE?RAM$"),           "SHRIRAM"),         # SHREERAM, SRIRAM, SHRRAM
]

print("brand aliases:", len(BRAND_ALIASES), "| phrases:", len(BRAND_ALIAS_PHRASES),
      "| regex folds:", len(BRAND_REGEX))

In [ ]:
# Step 2.4 : Token-level replacement pipeline
#
# Order: initial-run join -> bigram collapse -> brand phrases -> per-token OCR fix
#        -> per-token abbreviation expand -> per-token brand alias -> regex brand fold
#        -> repeated-n-gram dedup.
# Every step is exact-match on whole tokens, so village / person names are never rewritten.
_MAX_PHRASE = max(len(k) for k in BRAND_ALIAS_PHRASES)

def _apply_bigrams(toks):
    out, i = [], 0
    while i < len(toks):
        if i + 1 < len(toks) and (toks[i], toks[i + 1]) in ABBREV_BIGRAMS:
            out.extend(ABBREV_BIGRAMS[(toks[i], toks[i + 1])].split())
            i += 2
        else:
            out.append(toks[i])
            i += 1
    return out

def _apply_phrases(toks):
    out, i = [], 0
    while i < len(toks):
        hit = None
        for L in range(min(_MAX_PHRASE, len(toks) - i), 1, -1):
            key = tuple(toks[i:i + L])
            if key in BRAND_ALIAS_PHRASES:
                hit = (BRAND_ALIAS_PHRASES[key].split(), L)
                break
        if hit:
            out.extend(hit[0])
            i += hit[1]
        else:
            out.append(toks[i])
            i += 1
    return out

def _join_initial_runs(toks):
    '''"H D F C BANK" -> [("HDFC", True), ("BANK", False)].

    Punctuation stripping in Phase 1 turns "H.D.F.C." into four single-letter tokens; without
    this join no alias or gazetteer entry can ever match them. The flag marks a joined token as
    an initialism so the abbreviation pass skips it - otherwise "DR B R AMBEDKAR" would join to
    "BR" and then expand to "BRANCH".
    '''
    out, i = [], 0
    while i < len(toks):
        if len(toks[i]) == 1 and toks[i].isalpha():
            j = i
            while j < len(toks) and len(toks[j]) == 1 and toks[j].isalpha():
                j += 1
            if j - i >= 2:
                out.append(("".join(toks[i:j]), True))
                i = j
                continue
        out.append((toks[i], False))
        i += 1
    return out

def _dedup_ngrams(toks, max_n=4):
    '''Drop an n-gram that immediately repeats itself.

    Expansion can duplicate text: "HDFC BANK LTD" -> alias HDFC="HDFC BANK" -> "HDFC BANK BANK".
    Longer case: "AU SMALL FINANCE BANK" + "SMALL FINANCE BANK".
    '''
    for n in range(max_n, 0, -1):
        k = 0
        while k + 2 * n <= len(toks):
            if toks[k:k + n] == toks[k + n:k + 2 * n]:
                del toks[k + n:k + 2 * n]
            else:
                k += 1
    return toks

def normalize_typos(clean: str) -> str:
    flagged = _join_initial_runs(clean.split())

    # initialisms bypass OCR/abbrev; they only get brand-alias treatment
    staged, initial_flags = [], []
    for t, is_init in flagged:
        staged.append(t)
        initial_flags.append(is_init)

    staged = _apply_bigrams(staged)
    staged = _apply_phrases(staged)
    init_set = {t for t, f in zip([t for t, _ in flagged], [f for _, f in flagged]) if f}

    out = []
    for t in staged:
        if t in PROTECTED_TOKENS:
            out.append(t)
            continue
        if t not in init_set:                  # joined initialisms skip 1 and 2
            t = OCR_DICT.get(t, t)             # 1. spelling / OCR
            t = ABBREV_DICT.get(t, t)          # 2. abbreviation expansion (may yield 2 tokens)
        t = BRAND_ALIASES.get(t, t)            # 3. brand initialism
        for pat, repl in BRAND_REGEX:          # 4. regex brand fold
            if pat.match(t):
                t = repl
                break
        out.extend(t.split())

    return " ".join(_dedup_ngrams(out))

df["typo_fixed"] = df["clean_name"].progress_map(normalize_typos)
df[["clean_name", "typo_fixed"]].head(20)

In [ ]:
# Step 2.5 : Validation - show only rows the dictionaries actually changed (up to 200)
changed = df[df["clean_name"] != df["typo_fixed"]][["clean_name", "typo_fixed"]]
print("changed rows:", len(changed), f"({len(changed)/len(df):.1%})")
for a, b in changed.sample(min(200, len(changed)), random_state=11).itertuples(index=False):
    print(f"{a!r:55} -> {b!r}")

In [ ]:
# Step 2.6 : Validation - spot-check the abbreviation ladders explicitly
for probe in [
    "CHOLAMANDALAM INVE AND FIN COM LTD",
    "CHOLAMANDALAM INVESTMENT AND FINANCE COMPANY LIMITED",
    "S B I KARPOORI THAKUR SADAN BR",
    "SBI PB BR DHANBAD",
    "NAGBHID NAG SAH PATH SANSTHA",
    "PARIVARTAN GAR BIG SHE SAH PAT MAR",
    "THE UDR URBAN CO OP BANK LTD",
    "SHRRAM CTY UNION FIIN LTD",
    "C I F C L",
]:
    print(f"{probe!r:52} -> {normalize_typos(probe)!r}")

## Phase 3 : Institution Resolution (branch collapse)

**The financer-specific phase.** `rc_financer` is captured at the branch that financed the vehicle,
so the same institution appears with hundreds of location suffixes:

```
INDIAN BANK .JUNAGADH          \
INDIAN BANK, BORSUL BR.,BURDWAN >--> INDIAN BANK
STATE BANK OF INDIA HATHUR     /      STATE BANK OF INDIA
SBI, PB BR DHANBAD            /
```

The entity we want is the **institution**, so the branch/location residue is stripped from the
name used for clustering and parked in `branch_hint` for audit.

**Two mechanisms, in order:**

1. **Gazetteer anchor** (high precision). `INSTITUTIONS` holds the token sequences of scheduled
   banks, SFBs, RRBs and the large NBFCs. A name is matched by *longest token sequence starting at
   token 0* (leading `THE` / `M S` dropped first). On a hit, `institution_name` is the gazetteer's
   canonical form and everything left over becomes `branch_hint`. This is what makes
   `CENTRAL BANK OF INDIA` not collapse into `BANK OF INDIA` - the longer sequence wins at pos 0.
2. **Branch-marker split** (fallback, for co-operative societies which are not in the gazetteer).
   Everything from the first strong branch marker (`BRANCH`, `OPP`, `NEAR`, `AT POST`) onward moves
   to `branch_hint`; the head stays as the name. Weak markers alone (a bare trailing village name)
   are **not** stripped - there is no safe way to tell "SHIMOGA DISTRICT CO OPERATIVE BANK" (where
   SHIMOGA *is* the identity) from a trailing branch, so co-op names keep their locality.

In [ ]:
# Step 3.1 : Institution gazetteer
#
# Written in the POST-Phase-2 surface form (abbreviations already expanded), because matching runs
# on typo_fixed. Add an entry whenever a large financer shows up with branch noise.
INSTITUTION_LIST = [
    # ---- public sector banks ----
    "STATE BANK OF INDIA",
    "STATE BANK OF HYDERABAD", "STATE BANK OF PATIALA", "STATE BANK OF TRAVANCORE",
    "STATE BANK OF MYSORE", "STATE BANK OF BIKANER AND JAIPUR",
    "PUNJAB NATIONAL BANK", "BANK OF BARODA", "BANK OF INDIA", "BANK OF MAHARASHTRA",
    "CANARA BANK", "UNION BANK OF INDIA", "CENTRAL BANK OF INDIA", "INDIAN BANK",
    "INDIAN OVERSEAS BANK", "UCO BANK", "PUNJAB AND SIND BANK", "IDBI BANK",
    "ORIENTAL BANK OF COMMERCE", "ALLAHABAD BANK", "ANDHRA BANK", "CORPORATION BANK",
    "SYNDICATE BANK", "VIJAYA BANK", "DENA BANK", "UNITED BANK OF INDIA",

    # ---- private sector banks ----
    "HDFC BANK", "ICICI BANK", "AXIS BANK", "KOTAK MAHINDRA BANK", "INDUSIND BANK",
    "YES BANK", "IDFC FIRST BANK", "FEDERAL BANK", "SOUTH INDIAN BANK",
    "KARUR VYSYA BANK", "KARNATAKA BANK", "CITY UNION BANK", "TAMILNAD MERCANTILE BANK",
    "DHANLAXMI BANK", "RBL BANK", "BANDHAN BANK", "CSB BANK", "NAINITAL BANK",
    "JAMMU AND KASHMIR BANK", "DCB BANK", "LAKSHMI VILAS BANK", "CATHOLIC SYRIAN BANK",

    # ---- large urban co-operative banks (institution-level, they carry branch noise too) ----
    "SARASWAT COOPERATIVE BANK", "COSMOS COOPERATIVE BANK", "ABHYUDAYA COOPERATIVE BANK",
    "SHAMRAO VITHAL COOPERATIVE BANK", "BHARAT COOPERATIVE BANK", "TJSB SAHAKARI BANK",
    "NKGSB COOPERATIVE BANK", "JANATA SAHAKARI BANK", "KALUPUR COMMERCIAL COOPERATIVE BANK",
    "RAJKOT NAGARI SAHAKARI BANK", "MEHSANA URBAN COOPERATIVE BANK",

    # ---- small finance / payments banks ----
    "AU SMALL FINANCE BANK", "EQUITAS SMALL FINANCE BANK", "UJJIVAN SMALL FINANCE BANK",
    "ESAF SMALL FINANCE BANK", "SURYODAY SMALL FINANCE BANK", "JANA SMALL FINANCE BANK",
    "FINCARE SMALL FINANCE BANK", "UTKARSH SMALL FINANCE BANK", "NORTH EAST SMALL FINANCE BANK",
    "CAPITAL SMALL FINANCE BANK", "SHIVALIK SMALL FINANCE BANK", "UNITY SMALL FINANCE BANK",

    # ---- regional rural banks (the ones that carry branch noise heavily) ----
    "KERALA GRAMIN BANK", "KARNATAKA GRAMIN BANK", "KARNATAKA VIKAS GRAMIN BANK",
    "ANDHRA PRADESH GRAMEENA VIKAS BANK", "ASSAM GRAMIN VIKASH BANK",
    "DAKSHIN BIHAR GRAMIN BANK", "UTTAR BIHAR GRAMIN BANK", "MADHYA PRADESH GRAMIN BANK",
    "MADHYANCHAL GRAMIN BANK", "BARODA UP BANK", "BARODA RAJASTHAN KSHETRIYA GRAMIN BANK",
    "BARODA GUJARAT GRAMIN BANK", "UTTAR PRADESH GRAMIN BANK", "PRATHAMA UP GRAMIN BANK",
    "PASCHIM BANGA GRAMIN BANK", "BANGIYA GRAMIN VIKASH BANK", "TELANGANA GRAMEENA BANK",
    "SAPTAGIRI GRAMEENA BANK", "CHAITANYA GODAVARI GRAMEENA BANK",
    "TAMIL NADU GRAMA BANK", "PUDUVAI BHARATHIAR GRAMA BANK",
    "MAHARASHTRA GRAMIN BANK", "VIDHARBHA KONKAN GRAMIN BANK",
    "RAJASTHAN MARUDHARA GRAMIN BANK", "HIMACHAL PRADESH GRAMIN BANK",
    "JAMMU AND KASHMIR GRAMIN BANK", "ELLAQUAI DEHATI BANK", "PUNJAB GRAMIN BANK",
    "SARVA HARYANA GRAMIN BANK", "UTKAL GRAMEEN BANK", "ODISHA GRAMYA BANK",
    "TRIPURA GRAMIN BANK", "MANIPUR RURAL BANK", "MEGHALAYA RURAL BANK",
    "ARUNACHAL PRADESH RURAL BANK", "NAGALAND RURAL BANK", "MIZORAM RURAL BANK",
    "JHARKHAND RAJYA GRAMIN BANK", "CHHATTISGARH RAJYA GRAMIN BANK",
    "BIHAR GRAMIN BANK", "ODISHA GRAMIN BANK", "ASSAM GRAMIN BANK",
    "BARODA UTTAR PRADESH BANK", "BARODA UTTAR PRADESH GRAMIN BANK",
    "KASHI GOMTI SAMYUT GRAMIN BANK", "PURVANCHAL GRAMIN BANK",
    "ARYAVART BANK", "BARODA UTTAR PRADESH KSHETRIYA GRAMIN BANK",

    # ---- state co-operative / apex banks (institution-level, not society-level) ----
    "HIMACHAL PRADESH STATE COOPERATIVE BANK", "MAHARASHTRA STATE COOPERATIVE BANK",
    "GUJARAT STATE COOPERATIVE BANK", "KARNATAKA STATE COOPERATIVE APEX BANK",
    "KERALA STATE COOPERATIVE BANK", "PUNJAB STATE COOPERATIVE BANK",
    "RAJASTHAN STATE COOPERATIVE BANK", "TAMIL NADU STATE APEX COOPERATIVE BANK",

    # ---- NBFCs / captives ----
    "CHOLAMANDALAM INVESTMENT FINANCE",
    "MAHINDRA AND MAHINDRA FINANCIAL SERVICES", "MAHINDRA FINANCIAL SERVICES",
    "SHRIRAM FINANCE", "SHRIRAM TRANSPORT FINANCE", "SHRIRAM CITY UNION FINANCE",
    "BAJAJ FINANCE", "BAJAJ AUTO FINANCE", "BAJAJ FINSERV",
    "TATA MOTORS FINANCE", "TATA CAPITAL FINANCIAL SERVICES", "TATA CAPITAL",
    "TVS CREDIT SERVICES", "HDB FINANCIAL SERVICES", "SMFG INDIA CREDIT",
    "HERO FINCORP", "HINDUJA LEYLAND FINANCE", "ASHOK LEYLAND FINANCE",
    "SUNDARAM FINANCE", "MUTHOOT FINANCE", "MUTHOOT CAPITAL SERVICES",
    "MUTHOOT VEHICLE AND ASSET FINANCE", "MANAPPURAM FINANCE",
    "L AND T FINANCE", "MAGMA FINCORP", "POONAWALLA FINCORP",
    "ORIX AUTO INFRASTRUCTURE SERVICES", "ORIX LEASING AND FINANCIAL SERVICES",
    "TOYOTA FINANCIAL SERVICES INDIA", "DAIMLER FINANCIAL SERVICES INDIA",
    "VOLKSWAGEN FINANCE", "BMW INDIA FINANCIAL SERVICES", "MERCEDES BENZ FINANCIAL SERVICES",
    "JOHN DEERE FINANCIAL", "CNH INDUSTRIAL CAPITAL", "SREI EQUIPMENT FINANCE",
    "IIFL FINANCE", "AAVAS FINANCIERS", "MAS FINANCIAL SERVICES",
    "KOGTA FINANCIAL INDIA", "INDOSTAR CAPITAL FINANCE",
    "S K FINANCE", "CREDIT WISE CAPITAL", "WHEELS EMI", "BERAR FINANCE",
    "AEON CREDIT SERVICE INDIA", "NISSAN RENAULT FINANCIAL SERVICES INDIA",
    "BAJAJ AUTO CREDIT", "KOTAK MAHINDRA PRIME", "AXIS FINANCE",
    "VE COMMERCIAL VEHICLES", "SREI EQUIPMENT FINANCE", "ICICI HOME FINANCE",
    "LIC", "LIC HOUSING FINANCE",
]

# alias -> canonical (in addition to the canonical string matching itself)
INSTITUTION_ALIASES = {
    "SBI": "STATE BANK OF INDIA",
    "STATE BANK INDIA": "STATE BANK OF INDIA",
    "STATE BANK OF INDIA BANK": "STATE BANK OF INDIA",
    "BANK OF BARODA BANK": "BANK OF BARODA",
    "IDFC BANK": "IDFC FIRST BANK",
    "KOTAK BANK": "KOTAK MAHINDRA BANK",
    "KOTAK MAHINDRA PRIME": "KOTAK MAHINDRA PRIME",
    "J AND K BANK": "JAMMU AND KASHMIR BANK",
    "JAMMU AND KASHMIR BANK LIMITED": "JAMMU AND KASHMIR BANK",
    # Cholamandalam: every surface form is the same NBFC
    "CHOLAMANDALAM": "CHOLAMANDALAM INVESTMENT FINANCE",
    "CHOLAMANDALAM INVESTMENT AND FINANCE": "CHOLAMANDALAM INVESTMENT FINANCE",
    "CHOLAMANDALAM INVESTMENT FINANCE COMPANY": "CHOLAMANDALAM INVESTMENT FINANCE",
    "CHOLAMANDALAM FINANCE": "CHOLAMANDALAM INVESTMENT FINANCE",
    "CHOLAMANDALAM INVESTMENT": "CHOLAMANDALAM INVESTMENT FINANCE",
    # Mahindra
    "MAHINDRA AND MAHINDRA": "MAHINDRA AND MAHINDRA FINANCIAL SERVICES",
    "MAHINDRA AND MAHINDRA FINANCE": "MAHINDRA AND MAHINDRA FINANCIAL SERVICES",
    "MAHINDRA AND MAHINDRA FINANCE SERVICES": "MAHINDRA AND MAHINDRA FINANCIAL SERVICES",
    "MAHINDRA AND MAHINDRA FS": "MAHINDRA AND MAHINDRA FINANCIAL SERVICES",
    "MAHINDRA FINANCE": "MAHINDRA AND MAHINDRA FINANCIAL SERVICES",
    "MAHINDRA FINANCIAL SERVICES": "MAHINDRA AND MAHINDRA FINANCIAL SERVICES",
    "MAHINDRA FINANCE SERVICES": "MAHINDRA AND MAHINDRA FINANCIAL SERVICES",
    # Shriram
    "SHRIRAM TRANSPORT FINANCE COMPANY": "SHRIRAM TRANSPORT FINANCE",
    "SHRIRAM CITY UNION": "SHRIRAM CITY UNION FINANCE",
    "SHRIRAM CITY": "SHRIRAM CITY UNION FINANCE",
    "SHRIRAM TRANSPORT": "SHRIRAM TRANSPORT FINANCE",
    # L&T
    "L AND T FINANCIAL SERVICES": "L AND T FINANCE",
    "L AND T FINANCE HOLDINGS": "L AND T FINANCE",
    # ORIX
    "ORIX LEASING AND FINANCE SERVICES INDIA": "ORIX LEASING AND FINANCIAL SERVICES",
    "ORIX LEASING AND FINANCIAL SERVICES INDIA": "ORIX LEASING AND FINANCIAL SERVICES",
    "ORIX AUTO INFRASTRUCTURE": "ORIX AUTO INFRASTRUCTURE SERVICES",
    # AU
    "AU SMALL FINANCE": "AU SMALL FINANCE BANK",
    "AU BANK": "AU SMALL FINANCE BANK",
    "AU FINANCIERS": "AU SMALL FINANCE BANK",
    "AU FINANCIERS INDIA": "AU SMALL FINANCE BANK",
    # misc
    "HERO FINANCE CORPORATION": "HERO FINCORP",
    "HERO FINCORP FINANCE": "HERO FINCORP",
    "TATA MOTOR FINANCE": "TATA MOTORS FINANCE",
    "HDB FINANCIAL": "HDB FINANCIAL SERVICES",
    "TOYOTA FINANCIAL SERVICES": "TOYOTA FINANCIAL SERVICES INDIA",
    "HP STATE COOPERATIVE BANK": "HIMACHAL PRADESH STATE COOPERATIVE BANK",
    "UTTAR PRADESH GRAMIN BANK": "UTTAR PRADESH GRAMIN BANK",
    "BARODA UP BANK": "BARODA UTTAR PRADESH BANK",
    "MUTHOOT FINCORP": "MUTHOOT FINCORP",
    "SUNDARAM FINANCE HOLDINGS": "SUNDARAM FINANCE",
    "SK FINANCE": "S K FINANCE",
    "NISSAN RENAULT FINANCE": "NISSAN RENAULT FINANCIAL SERVICES INDIA",
    "NISSAN RENAULT FINANCIAL SERVICES": "NISSAN RENAULT FINANCIAL SERVICES INDIA",
    "AEON CREDIT SERVICES": "AEON CREDIT SERVICE INDIA",
    "AEON CREDIT SERVICES INDIA": "AEON CREDIT SERVICE INDIA",
    "ORIX LEASING AND FINANCE": "ORIX LEASING AND FINANCIAL SERVICES",
    "ORIX LEASING AND FINANCE SERVICES": "ORIX LEASING AND FINANCIAL SERVICES",
    "ORIX LEASING FINANCE SERVICES INDIA": "ORIX LEASING AND FINANCIAL SERVICES",
    "L AND T HOLDINGS FINANCE": "L AND T FINANCE",
    "L AND T FINANCIAL": "L AND T FINANCE",
    "L AND T FINANCIAL HOLDINGS": "L AND T FINANCE",
    # district central co-operative banks that appear as bare initialisms
    "KCC BANK": "KANGRA CENTRAL COOPERATIVE BANK",
    "KDCC BANK": "KDCC BANK",
    "SDCC BANK": "SDCC BANK",
    "SCDCC BANK": "SCDCC BANK",
    "SDC BANK": "SDCC BANK",
}

print("gazetteer entries:", len(INSTITUTION_LIST), "| aliases:", len(INSTITUTION_ALIASES))

In [ ]:
# Step 3.2 : Build the match index + institution resolver
#
# Longest-sequence-at-position-0 match. Leading THE / M S are dropped first so
# "THE HP STATE COOPERATIVE BANK LIMITED HATLI" still anchors.
_LEAD_DROP = {"THE", "M", "S", "MS"}

INST_INDEX = {}
for _canon in INSTITUTION_LIST:
    INST_INDEX[tuple(_canon.split())] = _canon
for _alias, _canon in INSTITUTION_ALIASES.items():
    INST_INDEX[tuple(_alias.split())] = _canon
_MAX_INST = max(len(k) for k in INST_INDEX)

# strong branch markers -> everything from here on is location, never identity
BRANCH_MARKERS = {
    "BRANCH", "OPP", "NEAR", "BEHIND", "ADB", "HUB", "RETAILASS", "RASECTION",
    "EXTENSION", "EXTN", "COUNTER", "OFFICE", "OFF",
}
# tokens that are pure address furniture; dropped from the tail of an unanchored name
ADDRESS_NOISE = {
    "ROAD", "RD", "CHOWK", "MAIN", "MARKET", "BAZAR", "BAZAAR", "MANDI", "TOWN",
    "CITY", "VILLAGE", "TALUKA", "TEHSIL", "POST", "PIN", "COMPLEX", "BUILDING",
}

def _drop_lead(toks):
    i = 0
    while i < len(toks) and toks[i] in _LEAD_DROP:
        i += 1
    return toks[i:], toks[:i]

def resolve_institution(name: str):
    '''(institution_name, branch_hint, anchored)

    anchored=True  -> matched the gazetteer; branch_hint is the location residue.
    anchored=False -> co-operative / unknown financer; only strong branch markers were split off.
    '''
    toks, dropped = _drop_lead(name.split())
    if not toks:
        return name, "", False

    # ---- 1. gazetteer anchor at position 0 ----
    for L in range(min(_MAX_INST, len(toks)), 0, -1):
        key = tuple(toks[:L])
        if key in INST_INDEX:
            canon    = INST_INDEX[key]
            residue  = toks[L:]
            # legal-form tokens right after the anchor are not branch info
            residue  = [t for t in residue
                        if t not in {"LIMITED", "PRIVATE", "COMPANY", "CO", "LTD", "PVT",
                                     "THE", "AND", "OF", "INDIA"}]
            return canon, " ".join(residue), True

    # ---- 2. branch-marker split (co-operatives, unknown financers) ----
    for k, t in enumerate(toks):
        if t in BRANCH_MARKERS:
            head = toks[:k]
            if head:                       # never let the marker eat the whole name
                return " ".join(dropped + head), " ".join(toks[k:]), False
            break

    # ---- 3. trailing address furniture ----
    end = len(toks)
    while end > 1 and toks[end - 1] in ADDRESS_NOISE:
        end -= 1
    if end < len(toks):
        return " ".join(dropped + toks[:end]), " ".join(toks[end:]), False

    return name, "", False

In [ ]:
# Step 3.3 : Apply institution resolution
_res = df["typo_fixed"].progress_map(resolve_institution)
df["institution_name"] = [r[0] for r in _res]
df["branch_hint"]      = [r[1] for r in _res]
df["anchored"]         = [r[2] for r in _res]

print("anchored to gazetteer:", int(df["anchored"].sum()), f"({df['anchored'].mean():.1%})")
print("rows with a branch_hint:", int((df["branch_hint"].str.len() > 0).sum()))
df.loc[df["anchored"], ["typo_fixed", "institution_name", "branch_hint"]].head(25)

In [ ]:
# Step 3.4 : Validation - branch collapse actually collapsed
#
# Each of these should print ONE institution_name across many raw spellings.
for probe in ["STATE BANK OF INDIA", "INDIAN BANK", "BANK OF BARODA", "CANARA BANK",
              "CHOLAMANDALAM", "HDFC BANK"]:
    sub = df[df["institution_name"] == probe]
    print(f"\n--- {probe} : {len(sub)} raw strings ---")
    for o, b in sub[["original_name", "branch_hint"]].head(12).itertuples(index=False):
        print(f"   {o!r:52} branch={b!r}")

In [ ]:
# Step 3.5 : Validation - what did NOT anchor (should be co-operative societies, not big banks)
unanchored = df.loc[~df["anchored"], "institution_name"]
print("unanchored rows:", len(unanchored))
print("\ntop unanchored heads (first 3 tokens) - anything bank-shaped here is a MISSING gazetteer entry:")
heads = unanchored.str.split().str[:3].str.join(" ").value_counts().head(60)
print(heads.to_string())

## Phase 4 : Canonical Name Generation
Remove legal-form words; **keep** business-line descriptors and geography.

`LEGAL_STOPWORDS` here includes the vernacular legal forms that the owner pipeline never sees:
`MARYADIT` (Marathi "limited"), `NIYAMIT` (Kannada "registered"), `REGISTERED`.

Deliberately **NOT** descriptors (they stay discriminative): `STATE`, `CENTRAL`, `NATIONAL`,
`UNION`, `INDIA`, `INDIAN`, `OVERSEAS`, `FIRST`, `NEW`. Without this,
`STATE BANK OF INDIA` / `BANK OF INDIA` / `CENTRAL BANK OF INDIA` / `UNION BANK OF INDIA` all
strip down to `BANK` and fuse into one cluster.

In [ ]:
# Step 4.1 : Legal-form stopwords (stripped) vs descriptors (KEPT)
LEGAL_STOPWORDS = {
    "PRIVATE", "LIMITED", "PRIVATELIMITED", "LTD", "PVT", "PVTLTD", "LLP", "LLC",
    "INC", "PLC", "CORP", "CORPORATION", "INCORPORATED", "COMPANY", "CO", "OPC",
    "HUF", "LT", "PROPRIETOR", "PROPRIETORSHIP", "PROP", "PARTNER", "PARTNERS",
    "MANAGING", "DIRECTOR", "MR", "MRS", "MS", "M/S", "AND", "&", "THE", "OF", "FOR",
    # vernacular legal forms - "limited" / "registered"
    "MARYADIT", "NIYAMIT", "REGISTERED", "REGD",
    # branch furniture that survived Phase 3 (unanchored rows keep locality, not these)
    "BRANCH",
}

# KEPT in canonical_name. Weak identity on their own -> Phase 8 discounts them in core_tokens.
DESCRIPTOR_WORDS = {
    # finance business lines
    "BANK", "FINANCE", "FINANCIAL", "FINSERV", "FINCORP", "FINANCIERS", "CREDIT",
    "INVESTMENT", "CAPITAL", "LEASING", "HIRE", "PURCHASE", "NIDHI", "CHIT", "FUND",
    "FUNDS", "SECURITIES", "HOLDINGS", "ENTERPRISES", "SERVICES", "SOLUTIONS",
    "TRADERS", "TRADING", "AGENCIES", "AGENCY", "ASSOCIATES", "MOTORS", "AUTO",
    "AUTOMOBILES", "VEHICLE", "TRANSPORT", "EQUIPMENT", "ASSET", "HOUSING",
    "INFRASTRUCTURE", "INDUSTRIAL", "INDUSTRIES", "TECHNOLOGIES", "FINTECH",
    # co-operative / society structure
    "COOPERATIVE", "SOCIETY", "SAHAKARI", "SOUHARDA", "PATSANSTHA", "SANSTHA", "SANGHA",
    "BIGARSHETI", "SHETI", "MULTIPURPOSE", "MULTISTATE", "PRIMARY", "APEX", "URBAN",
    "NAGARI", "GRAMIN", "GRAMA", "RURAL", "MAHILA", "VYAPARI", "JANATA", "SEVA",
    "MERCANTILE", "MERCHANTS", "PEOPLES", "AGRI", "DEVELOPMENT", "VIKAS", "VIKASH",
    "SMALL", "PAYMENTS", "KSHETRIYA", "RAJYA", "DISTRICT",
}

assert not (LEGAL_STOPWORDS & DESCRIPTOR_WORDS), "a word must be legal-form OR descriptor, not both"

# Explicitly discriminative - documented so nobody 'tidies' them into DESCRIPTOR_WORDS later.
KEEP_DISCRIMINATIVE = {"STATE", "CENTRAL", "NATIONAL", "UNION", "INDIA", "INDIAN",
                       "OVERSEAS", "FIRST", "NEW", "SOUTH", "NORTH", "EAST", "WEST"}
assert not (KEEP_DISCRIMINATIVE & DESCRIPTOR_WORDS), "these words separate real institutions"
assert not (KEEP_DISCRIMINATIVE & LEGAL_STOPWORDS)

print("legal stopwords:", len(LEGAL_STOPWORDS), "| descriptors kept:", len(DESCRIPTOR_WORDS))

In [ ]:
# Step 4.2 : Canonicalization
def canonicalize(name: str) -> str:
    toks = [t for t in name.split() if t not in LEGAL_STOPWORDS]
    canon = " ".join(toks).strip()
    # guard: never return empty -> fall back to the un-stripped name
    return canon if canon else name

df["canonical_name"] = df["institution_name"].map(canonicalize)

# Final schema: original_name, clean_name, typo_fixed, institution_name, branch_hint,
#               anchored, canonical_name
df[["original_name", "canonical_name", "branch_hint"]].head(15)

In [ ]:
# Step 4.3 : Validation - original -> canonical for 200 random rows
for o, c, b in df.sample(200, random_state=7)[
        ["original_name", "canonical_name", "branch_hint"]].itertuples(index=False):
    print(f"{o!r:52} -> {c!r:42} | branch={b!r}")

In [ ]:
# Step 4.4 : Validation - the four *BANK OF INDIA institutions must stay separate
for probe in ["STATE BANK OF INDIA", "BANK OF INDIA", "CENTRAL BANK OF INDIA",
              "UNION BANK OF INDIA"]:
    print(f"{probe!r:26} -> canonical {canonicalize(probe)!r}")

In [ ]:
# Step 4.5 : Persist cleaned frame + drop rows with empty canonical
df = df[df["canonical_name"].str.len() > 0].reset_index(drop=True)
df["row_id"] = np.arange(len(df))          # stable id used everywhere downstream
df.to_parquet(art("names_clean.parquet"), index=False)

print("kept rows:", len(df))
print("distinct canonical names:", df["canonical_name"].nunique(),
      f"-> {df['canonical_name'].nunique()/len(df):.1%} of rows need an embedding")

## Phase 5 : Embedding Generation
Encode `canonical_name` with the Qwen embedding model, L2-normalized.

**Difference from the owner pipeline**: only the **distinct** canonical names are embedded, then
the vectors are mapped back to rows. Institution collapse turns hundreds of branch strings into one
canonical name, so this cuts the encode cost and the FAISS index size by a large factor.

In [ ]:
# Step 5.1 : Load Qwen embedding model (GPU only)
import torch
from sentence_transformers import SentenceTransformer

assert torch.cuda.is_available(), "CUDA GPU not available - this pipeline requires a GPU"
model = SentenceTransformer(CONFIG["model_name"], device="cuda")   # downloads on first run
print("device:", model.device, "| embedding dim:", model.get_sentence_embedding_dimension())

In [ ]:
# Step 5.2 : Encode the DISTINCT canonical names, then expand back to rows
uniq_names = df["canonical_name"].drop_duplicates().reset_index(drop=True)
name_to_uid = {n: k for k, n in enumerate(uniq_names)}
df["uid"] = df["canonical_name"].map(name_to_uid).astype("int64")

print("rows:", len(df), "| distinct names to encode:", len(uniq_names))

uniq_emb = model.encode(
    uniq_names.tolist(),
    batch_size=CONFIG["batch_size"],
    normalize_embeddings=CONFIG["normalize"],   # L2 norm == 1, so inner product == cosine
    show_progress_bar=True,
    convert_to_numpy=True,
    device="cuda",
).astype("float32")

# persist uid alongside the cleaned frame - a kernel restart would otherwise lose it
df.to_parquet(art("names_clean.parquet"), index=False)

np.save(art("uniq_embeddings.npy"), uniq_emb)
uniq_names.to_frame("canonical_name").to_parquet(art("uniq_names.parquet"), index=False)
print("uniq embeddings:", uniq_emb.shape)

In [ ]:
# Step 5.3 : Validation - shape + L2 norm ~= 1
uniq_emb = np.load(art("uniq_embeddings.npy"))
norms = np.linalg.norm(uniq_emb, axis=1)
print("shape:", uniq_emb.shape)
print("norm min/max:", norms.min(), norms.max())   # expect ~1.0 both
assert uniq_emb.shape[0] == df["uid"].nunique()

## Phase 6 : FAISS Index
Build `IndexFlatIP` (exact inner-product = cosine on normalized vectors) over the **distinct**
canonical names. Search top-100 neighbors.

If `uniq_emb` is still too large for exact search, swap in
`faiss.index_factory(dim, "IVF4096,Flat", faiss.METRIC_INNER_PRODUCT)` and train it - the rule
engine is precision-first, so a small recall loss at this stage is acceptable.

In [ ]:
# Step 6.1 : Build index
import faiss

dim = uniq_emb.shape[1]
index = faiss.IndexFlatIP(dim)     # exact search, cosine via normalized IP
index.add(uniq_emb)
print("indexed vectors:", index.ntotal)

faiss.write_index(index, art("faiss_flatip.index"))

In [ ]:
# Step 6.2 : Search top-K neighbors for every distinct name
K = CONFIG["top_k"]
scores, neighbors = index.search(uniq_emb, K)   # both shape (U, K)

np.save(art("faiss_scores.npy"), scores)
np.save(art("faiss_neighbors.npy"), neighbors)
print("scores:", scores.shape, "| neighbors:", neighbors.shape)

In [ ]:
# Step 6.3 : Validation - self-similarity ~= 1 for 50 random names
uniq_names = pd.read_parquet(art("uniq_names.parquet"))["canonical_name"]
for i in np.random.RandomState(0).randint(0, len(uniq_names), 50):
    self_pos = np.where(neighbors[i] == i)[0]
    s = scores[i][self_pos[0]] if len(self_pos) else float("nan")
    print(f"uid {i:>7}  self_sim={s:.4f}  {uniq_names.iloc[i]!r}")

## Phase 7 : Candidate Pair Generation
Turn neighbor lists into unique undirected candidate pairs over **uids** (distinct canonical names).
Drop self-matches and duplicate `(i,j)`/`(j,i)` edges. Apply a coarse cosine gate to cut noise.

In [ ]:
# Step 7.1 : Build unique (i, j, cosine) candidate pairs
gate = CONFIG["rules"]["min_cosine_gate"]
seen = set()
pair_i, pair_j, pair_cos = [], [], []

for i in tqdm(range(neighbors.shape[0]), desc="pairs"):
    for rank in range(neighbors.shape[1]):
        j = int(neighbors[i, rank])
        c = float(scores[i, rank])
        if j == i:            # skip self-match
            continue
        if c < gate:          # coarse cosine gate -> prune obvious non-matches early
            continue
        a, b = (i, j) if i < j else (j, i)   # canonical ordering -> dedup (i,j)==(j,i)
        if (a, b) in seen:
            continue
        seen.add((a, b))
        pair_i.append(a); pair_j.append(b); pair_cos.append(c)

pairs = pd.DataFrame({"i": pair_i, "j": pair_j, "cosine": pair_cos})
del seen
print("candidate pairs:", len(pairs))
pairs.head()

In [ ]:
# Step 7.2 : Validation - no self, no duplicate undirected edge
assert (pairs["i"] != pairs["j"]).all(), "self-match leaked"
assert not pairs.duplicated(subset=["i", "j"]).any(), "duplicate edge leaked"
assert (pairs["i"] < pairs["j"]).all(), "ordering broken"
print("candidate pairs OK:", len(pairs))

## Phase 8 : Feature Engineering
For every candidate pair compute string + token + semantic features.

Owner-pipeline features carried over: `cosine`, `rapidfuzz`, `token_sort`, `token_set`,
`levenshtein`, `jaccard`, `common_tokens`, `prefix_match`, `biz_kw_match`, `core_shared`,
`core_sim`, `single_token`, `core_empty`, `desc_conflict`.

**New, financer-specific:**
- **`both_anchored`** - both sides resolved to a gazetteer institution.
- **`inst_conflict`** - both anchored but to *different* institutions. Hardest possible reject:
  the gazetteer already proved they are two different banks (`BANK OF INDIA` vs
  `CENTRAL BANK OF INDIA` sit at cosine ~0.97).
- **`same_inst`** - both anchored to the same institution. Hardest possible accept.
- **`geo_conflict`** - the two names carry different state/region words
  (`KERALA GRAMIN BANK` vs `KARNATAKA GRAMIN BANK`), which is the co-operative analogue of
  `desc_conflict`.

In [ ]:
# Step 8.1 : Generic-word set used by the features, and token helpers
#
# WEAK_TOKENS / DISTRICT_WORDS affect core_tokens, strip_business and geo_tokens ONLY.
# They do NOT touch canonical_name, so editing them never invalidates the embeddings,
# the FAISS index or the candidate pairs.
BUSINESS_KEYWORDS = LEGAL_STOPWORDS | DESCRIPTOR_WORDS

# Honorifics, devotional prefixes, hypothecation furniture, vernacular society structure.
# None of these is identity: thousands of unrelated credit societies start with SHRI, and
# before this set every one of them shared a "core" token with every other one.
WEAK_TOKENS = {
    # honorifics & devotional
    "SHRI", "SHREE", "SHRE", "SRI", "SHRIMAN", "SHRIMATI", "SANT", "SWAMI", "SWAMY",
    "MAHARAJ", "MAHARAJA", "BABA", "BAPU", "MATA", "MAA", "DEVI", "BHAGWAN", "JAI", "OM",
    # hypothecation / lien furniture that survives cleaning
    "HYP", "HYPO", "HYPT", "HYPOTHECATED", "HYPOTHECATION", "HP", "HPA", "HPN", "HPT",
    "LOAN", "LOANS", "FINANCED", "UNDER", "VIDE", "THRU", "THROUGH", "VIA", "AT", "BY", "OR",
    "TE", "TH", "THR", "THE", "NO", "NOS",
    # co-operative structure words the Phase 4 descriptor list does not carry
    "PATHSANSTHA", "PATSANTHA", "PATPEDHI", "PEDHI", "SAHKARI", "SAHAKAR", "SOUHARDHA",
    "SOUHARD", "SOCIETIES", "NAGRI", "GRAMEEN", "VIVIDH", "KARYAKARI", "VYAVASAYIK",
    "SHETKARI", "PATTAN", "PAT", "SANSTH", "SANSTHAN",
}
BUSINESS_KEYWORDS = BUSINESS_KEYWORDS | WEAK_TOKENS

# State / region words. Two societies that differ only here are different financers.
GEO_WORDS = {
    "ANDHRA", "ARUNACHAL", "ASSAM", "BIHAR", "CHHATTISGARH", "GOA", "GUJARAT", "HARYANA",
    "HIMACHAL", "JHARKHAND", "KARNATAKA", "KERALA", "MADHYA", "MAHARASHTRA", "MANIPUR",
    "MEGHALAYA", "MIZORAM", "NAGALAND", "ODISHA", "ORISSA", "PUNJAB", "RAJASTHAN", "SIKKIM",
    "TAMIL", "TAMILNADU", "TELANGANA", "TRIPURA", "UTTARAKHAND", "PRADESH", "BENGAL",
    "JAMMU", "KASHMIR", "DELHI", "PUDUCHERRY", "MARATHWADA", "VIDHARBHA", "KONKAN",
    "MALWA", "SAURASHTRA", "MARUDHARA", "GODAVARI", "KRISHNA", "DAKSHIN", "UTTAR",
    "PASCHIM", "PURVA", "MADHYANCHAL", "BARODA", "MYSORE", "HYDERABAD", "PATIALA",
    "TRAVANCORE", "BIKANER", "JAIPUR",
}

# District / city names. District Central Co-operative Banks and credit societies differ ONLY
# by locality and cosine cannot see it: AKOLA WASHIM DCC BANK vs PUNE DCC BANK sit at ~0.96.
DISTRICT_WORDS = {
    # Maharashtra
    "AKOLA", "WASHIM", "AMRAVATI", "NAGPUR", "WARDHA", "CHANDRAPUR", "GADCHIROLI",
    "BHANDARA", "GONDIA", "YAVATMAL", "NANDED", "LATUR", "OSMANABAD", "BEED", "PARBHANI",
    "HINGOLI", "JALNA", "AURANGABAD", "NASHIK", "NASIK", "DHULE", "NANDURBAR", "JALGAON",
    "AHMEDNAGAR", "PUNE", "SATARA", "SANGLI", "SOLAPUR", "KOLHAPUR", "RATNAGIRI",
    "SINDHUDURG", "RAIGAD", "THANE", "PALGHAR", "MUMBAI", "ICHALKARANJI", "MALEGAON",
    # Karnataka
    "BELAGAVI", "BELGAUM", "BAGALKOT", "VIJAYAPURA", "BIJAPUR", "KALABURAGI", "GULBARGA",
    "BIDAR", "RAICHUR", "KOPPAL", "GADAG", "DHARWAD", "HUBLI", "HAVERI", "KANNADA",
    "UDUPI", "SHIVAMOGGA", "SHIMOGA", "CHITRADURGA", "DAVANAGERE", "TUMKUR", "KOLAR",
    "MANDYA", "MYSURU", "HASSAN", "CHIKMAGALUR", "CHAMARAJANAGAR", "BENGALURU",
    "BANGALORE", "YADGIR", "BALLARI", "BELLARY", "MANGALORE", "KANARA", "KARWAR",
    # Gujarat
    "AHMEDABAD", "SURAT", "VADODARA", "RAJKOT", "BHAVNAGAR", "JAMNAGAR", "JUNAGADH",
    "KUTCH", "KACHCHH", "MEHSANA", "PATAN", "BANASKANTHA", "SABARKANTHA", "KHEDA",
    "ANAND", "BHARUCH", "NAVSARI", "VALSAD", "AMRELI", "SURENDRANAGAR",
    # Rajasthan / North / Central
    "JODHPUR", "UDAIPUR", "KOTA", "AJMER", "ALWAR", "BHILWARA", "SIKAR", "LUDHIANA",
    "AMRITSAR", "JALANDHAR", "PATNA", "LUCKNOW", "KANPUR", "VARANASI", "AGRA", "MEERUT",
    "GORAKHPUR", "INDORE", "BHOPAL", "JABALPUR", "GWALIOR", "RAIPUR", "BILASPUR",
    # South
    "WARANGAL", "GUNTUR", "NELLORE", "CHITTOOR", "KURNOOL", "ANANTAPUR", "KADAPA",
    "VISAKHAPATNAM", "VIJAYAWADA", "TIRUPATI", "COIMBATORE", "MADURAI", "SALEM",
    "TIRUCHIRAPPALLI", "TRICHY", "ERODE", "VELLORE", "THANJAVUR", "TIRUNELVELI",
    "KANYAKUMARI", "CHENNAI", "ERNAKULAM", "KOCHI", "KOTTAYAM", "THRISSUR", "PALAKKAD",
    "KOZHIKODE", "KANNUR", "MALAPPURAM", "ALAPPUZHA", "KOLLAM", "TRIVANDRUM",
    "THIRUVANANTHAPURAM", "IDUKKI", "WAYANAD", "PATHANAMTHITTA", "KASARAGOD",
    # East
    "KOLKATA", "HOWRAH", "SILIGURI", "GUWAHATI", "CUTTACK", "BHUBANESWAR", "SAMBALPUR",
    "ROURKELA", "RANCHI", "JAMSHEDPUR", "DHANBAD",
}
GEO_WORDS = GEO_WORDS | DISTRICT_WORDS


from functools import lru_cache

# generic words bucketed by length, so the fuzzy check below only compares plausible candidates
_KW_BY_LEN = defaultdict(set)
for _k in BUSINESS_KEYWORDS:
    _KW_BY_LEN[len(_k)].add(_k)


def _fold(tok):
    # light singular/plural fold so SERVICE == SERVICES when comparing tokens
    return tok[:-1] if len(tok) > 4 and tok.endswith("S") else tok


# 4-letter generics allowed to absorb a 1-edit typo. Deliberately tiny: at length 4 a single
# edit reaches real brands (TATA is one edit from the honorific MATA), so only the words whose
# misspellings actually flood the data are listed here.
SHORT_FUZZY_GENERIC = {"BANK", "FUND", "LOAN"}


@lru_cache(maxsize=1_000_000)
def is_generic_token(t):
    '''Generic word, including a MISSPELLED one.

    "TVS CREDIT SIRVICES" and "FEDERAL BAMK" must not treat SIRVICES / BAMK as identity:
    a one-edit neighbour of a generic word is still that generic word.

    Guard: tokens shorter than 5 chars only fuzzy-match SHORT_FUZZY_GENERIC, otherwise brand
    tokens (TATA, BAJA, DENA) get stripped and their clusters collapse into core_empty.
    '''
    if t in BUSINESS_KEYWORDS:
        return True
    if len(t) < 4:
        return False
    if len(t) == 4:
        return any(Levenshtein.distance(t, k) <= 1 for k in SHORT_FUZZY_GENERIC)
    for L in (len(t) - 1, len(t), len(t) + 1):
        for k in _KW_BY_LEN.get(L, ()):
            if Levenshtein.distance(t, k) <= 1:
                return True
    return False


def _is_noise_token(t):
    # generic/weak word (or a typo of one), bare number ("19 HDFC BANK"), or single character
    return t.isdigit() or len(t) <= 1 or is_generic_token(t)

def core_tokens(name):
    # discriminative tokens only: drop generic/weak words, bare digits AND single-char initials
    return {_fold(t) for t in name.split() if not _is_noise_token(t)}

def strip_business(name):
    # remove generic words and bare digits but KEEP initials -> the distinguishing part
    return " ".join(t for t in name.split() if not (t.isdigit() or is_generic_token(t)))


def fuzzy_core_shared(ca, cb):
    '''Core tokens shared allowing one typo: AXIS ~ AXISE, MUTHOOT ~ MUTHOOTH.

    Greedy one-to-one match from the smaller set, so a single token can never count twice.
    '''
    small, large = (ca, cb) if len(ca) <= len(cb) else (cb, ca)
    free, hits = set(large), 0
    for t in small:
        if t in free:
            free.discard(t)
            hits += 1
            continue
        m = next((u for u in free
                  if min(len(t), len(u)) >= 4 and Levenshtein.distance(t, u) <= 1), None)
        if m is not None:
            free.discard(m)
            hits += 1
    return hits

def is_single_token(name):
    return len(name.split()) == 1

def descriptor_tokens(name):
    return {t for t in name.split() if t in DESCRIPTOR_WORDS}

def geo_tokens(name):
    return {t for t in name.split() if t in GEO_WORDS}


def _descriptors_compatible(x, y):
    '''Two descriptor words describe the same business line?

    Identical, one an abbreviation/prefix of the other (FIN/FINANCE, INVEST/INVESTMENT),
    or plainly similar (FINANCE/FINANCIAL). BANK vs MOTORS is neither.
    '''
    if x == y:
        return True
    short, long_ = (x, y) if len(x) <= len(y) else (y, x)
    if len(short) >= 3 and long_.startswith(short):
        return True
    return fuzz.ratio(x, y) >= 80


def descriptors_conflict(a, b):
    '''True when both names carry descriptors and NONE of them line up.

    "TATA MOTORS FINANCE" vs "TATA CAPITAL" -> same brand, different arm -> different financers.
    Returns False if either side has no descriptor at all (nothing to compare).
    '''
    da, db = descriptor_tokens(a), descriptor_tokens(b)
    if not da or not db:
        return False
    return not any(_descriptors_compatible(x, y) for x in da for y in db)


def geos_conflict(a, b):
    '''Both sides name a state/region/district and the sets are disjoint -> different bodies.'''
    ga, gb = geo_tokens(a), geo_tokens(b)
    if not ga or not gb:
        return False
    return not (ga & gb)

print("business keywords:", len(BUSINESS_KEYWORDS), "| geo words:", len(GEO_WORDS))
for n in ["STATE BANK INDIA", "SHRI BALGANESH MP COOPERATIVE SOCIETY",
          "SHRI MAHALAXMI PATTAN SAHAKARI", "19 HDFC BANK", "HYPO AXIS BANK"]:
    print(f"  {n!r:46} core={sorted(core_tokens(n))}")
print("  AKOLA DCC vs PUNE DCC geo_conflict:",
      geos_conflict("AKOLA WASHIM DISTRICT CENTRAL BANK",
                    "PUNE DISTRICT CENTRAL COOPERATIVE BANK"))
print("  TATA MOTORS FINANCE vs TATA CAPITAL desc_conflict:",
      descriptors_conflict("TATA MOTORS FINANCE", "TATA CAPITAL"))


In [ ]:
# Step 8.2 : Per-pair feature computation
def pair_features(a: str, b: str, cosine: float,
                  inst_a: str = "", inst_b: str = "",
                  anch_a: bool = False, anch_b: bool = False) -> dict:
    ta, tb = set(a.split()), set(b.split())
    inter, union = ta & tb, ta | tb

    ca, cb = core_tokens(a), core_tokens(b)     # discriminative tokens
    core_inter = ca & cb
    # typo-tolerant overlap: AXIS BANK vs AXISE BANK shares its brand token, exact set math
    # does not see it. Guards below use the fuzzy count; core_shared_exact is kept for audit.
    shared = fuzzy_core_shared(ca, cb)
    core_denom = len(ca) + len(cb) - shared
    small, large = (ca, cb) if len(ca) <= len(cb) else (cb, ca)

    # similarity of the DISTINGUISHING part (generic words removed, initials kept)
    #
    # token_set_ratio is deliberately NOT used here: it returns 100 for ANY subset match, so
    # "SHRI MAHALAXMI" scored 100 against "SHRI MAHALAXMI PATTAN SAHAKARI ..." and R3 chained
    # thousands of unrelated societies into one component.
    sa, sb = strip_business(a), strip_business(b)
    core_sim = max(
        fuzz.token_sort_ratio(sa, sb),
        fuzz.ratio(sa.replace(" ", ""), sb.replace(" ", "")),   # catches concatenation
    )

    both_anchored = int(anch_a and anch_b)

    return {
        "cosine":        cosine,
        "rapidfuzz":     fuzz.ratio(a, b),
        "token_sort":    fuzz.token_sort_ratio(a, b),
        "token_set":     fuzz.token_set_ratio(a, b),
        "levenshtein":   Levenshtein.distance(a, b),
        "jaccard":       (len(inter) / len(union)) if union else 0.0,
        "common_tokens": len(inter),
        "prefix_match":  int(a[:3] == b[:3]),
        "biz_kw_match":  int(len(inter) > 0 and len(core_inter) == 0),
        "core_shared":   shared,
        "core_shared_exact": len(core_inter),
        "core_sim":      core_sim,
        "single_token":  int(is_single_token(a) or is_single_token(b)),
        "core_empty":    int(len(ca) == 0 or len(cb) == 0),
        "desc_conflict": int(descriptors_conflict(a, b)),
        # ---- anti-chaining shape of the two core sets ----
        "core_jaccard":  (shared / core_denom) if core_denom else 0.0,
        "core_min":      min(len(ca), len(cb)),
        "core_subset":   int(bool(small) and shared == len(small)),
        "core_extra":    len(large) - len(small),
        # ---- financer-specific ----
        "both_anchored": both_anchored,
        "same_inst":     int(both_anchored and inst_a == inst_b),
        "inst_conflict": int(both_anchored and inst_a != inst_b),
        "geo_conflict":  int(geos_conflict(a, b)),
    }


In [ ]:
# Step 8.3 : Build feature table (over uids)
#
# Institution / anchor flags are per-uid: a canonical name maps to exactly one institution_name,
# so take the first row per uid.
uid_meta = (df.sort_values("row_id")
              .drop_duplicates("uid")
              .set_index("uid")[["canonical_name", "institution_name", "anchored"]]
              .sort_index())
canon = uid_meta["canonical_name"].to_numpy()
inst  = uid_meta["institution_name"].to_numpy()
anch  = uid_meta["anchored"].to_numpy()

feat_rows = []
for i, j, c in tqdm(pairs.itertuples(index=False), total=len(pairs), desc="features"):
    feat_rows.append(pair_features(canon[i], canon[j], c,
                                   inst[i], inst[j], bool(anch[i]), bool(anch[j])))

features = pd.DataFrame(feat_rows)
features.insert(0, "i", pairs["i"].values)
features.insert(1, "j", pairs["j"].values)
features.insert(2, "financer1", canon[pairs["i"].values])
features.insert(3, "financer2", canon[pairs["j"].values])

features.to_parquet(art("features.parquet"), index=False)
features.head(20)

In [ ]:
# Step 8.4 : Validation - inspect 500 random pairs manually
features.sample(500, random_state=1)[
    ["financer1", "financer2", "cosine", "token_set", "rapidfuzz", "levenshtein",
     "core_sim", "core_shared", "single_token", "same_inst", "inst_conflict", "geo_conflict"]
]

## Phase 9 : Rule Engine
Never merge on cosine alone. **Guards run before any merge path.**

**Order (first match wins):**

1. **`same_inst` accept** - both sides anchored to the same gazetteer institution. The gazetteer
   already proved identity; this is the path that fuses every branch spelling of a big bank.
2. **`inst_conflict` reject** - both anchored, different institutions. Nothing overrides this.
   Kills `BANK OF INDIA` vs `CENTRAL BANK OF INDIA` (cosine ~0.97).
3. **`geo_conflict` reject** - different state/region words. Kills `KERALA GRAMIN BANK` vs
   `KARNATAKA GRAMIN BANK`, `BARODA UP BANK` vs `BARODA GUJARAT GRAMIN BANK`.
4. **single-token guard** - one-word names merge only if identical.
5. **core-empty guard** - no discriminative token on either side (`URBAN COOPERATIVE BANK`) ->
   merge only on a character-identical canonical.
6. **`desc_conflict` reject** - shared brand, unrelated arm (`TATA MOTORS FINANCE` vs `TATA CAPITAL`).
7. near-exact bypass, then the owner pipeline's Rules 1-3.

In [ ]:
# Step 9.0 : RESUME from saved files (run in a FRESH kernel, after Step 0)
# Phase 9 onward uses these saved files, not upstream cell variables.
#
# Also execute these def-only cells first: 1.2 clean_name | 2.1-2.4 normalize_typos
# | 3.1-3.2 resolve_institution | 4.1-4.2 canonicalize | 8.1-8.2 features.
df       = pd.read_parquet(art("names_clean.parquet"))
features = pd.read_parquet(art("features.parquet"))

for _c in ["same_inst", "inst_conflict", "geo_conflict", "core_empty", "desc_conflict",
           "core_jaccard", "core_min", "core_subset", "core_extra"]:
    assert _c in features.columns, f"features.parquet is stale - re-run Phases 1-8 (missing {_c})"

print("rows:", len(df), "| pairs:", len(features))

In [ ]:
# Step 9.1 : Decision function per pair (precision-first: reject before merge)
#
# decide_reason() returns an audit label so Step 9.2 can show WHICH rule fired.
# decide() is the boolean wrapper used by Phase 10 and by the incremental resolver (Step 12.1).
R = CONFIG["rules"]

def decide_reason(f) -> str:
    # ---- Gazetteer verdicts: strongest signal available, checked first ----
    # (0) same institution proven by the gazetteer -> merge (branch spellings of one bank)
    if f["same_inst"] and f["cosine"] >= R["r4_inst_cos"]:
        return "accept:same_inst"
    # (1) different institutions proven by the gazetteer -> never merge
    if f["inst_conflict"]:
        return "reject:inst_conflict"        # BANK OF INDIA vs CENTRAL BANK OF INDIA

    # ---- Guards that no similarity score may override ----
    # (2) different state / region / district -> different institution
    if f["geo_conflict"]:
        return "reject:geo_conflict"         # KERALA vs KARNATAKA GRAMIN, AKOLA vs PUNE DCC
    # (3) single-token person/brand names -> merge only if identical
    if f["single_token"] and f["levenshtein"] != 0:
        return "reject:single_token"
    # (4) no discriminative token on either side (generic words only)
    if f["core_empty"]:
        return "accept:core_empty_identical" if f["levenshtein"] == 0 else "reject:core_empty"
    # (5) shared brand but disjoint business lines
    if f["desc_conflict"]:
        return "reject:desc_conflict"        # TATA MOTORS FINANCE vs TATA CAPITAL

    # ---- Anti-chaining guards: the distinguishing parts must genuinely agree ----
    # (6) both sides carry >= 2 discriminative tokens -> the cores must actually overlap
    if f["core_min"] >= 2 and f["core_jaccard"] < R["core_jaccard_min"]:
        return "reject:core_jaccard"
    # (7) one core is a strict subset of the other with several extra tokens. For unanchored
    #     names that is a DIFFERENT body, not a spelling variant of the same one.
    if (not f["both_anchored"]) and f["core_subset"] and f["core_extra"] >= R["core_extra_max"]:
        return "reject:core_subset"

    # near-exact full string -> same name, different spacing/concatenation
    if f["rapidfuzz"] >= R["near_exact_rf"] and f["cosine"] >= R["near_exact_cos"]:
        return "accept:near_exact"

    # ---- Hard rejects ----
    if f["core_sim"] < R["core_sim_min"]:
        return "reject:core_sim"             # distinguishing part too different
    if f["core_shared"] == 0:
        return "reject:no_shared_core"       # no shared brand/place token at all
    # (8) unanchored, both multi-core -> one shared token is a coincidence, demand two
    if (not f["both_anchored"]) and f["core_min"] >= 2 and f["core_shared"] < 2:
        return "reject:one_shared_core"

    # ---- Merge rules ----
    if f["cosine"] >= R["r1_cosine"] and f["token_set"] >= R["r1_token_set"] and f["rapidfuzz"] >= R["r1_rapidfuzz"]:
        return "accept:r1"
    if f["levenshtein"] <= R["r2_lev"] and f["cosine"] >= R["r2_cosine"]:
        return "accept:r2"
    # abbreviation expansion around a shared token - R3 is the chaining-prone rule, so an
    # unanchored pair has to clear a higher cosine bar than a gazetteer-anchored one.
    r3_cos = R["r3_cosine"] if f["both_anchored"] else R["r3_cosine_unanchored"]
    if f["cosine"] >= r3_cos and f["core_sim"] >= R["r3_core_sim"]:
        return "accept:r3"
    return "reject:no_rule"


def decide(f) -> bool:
    return decide_reason(f).startswith("accept")


In [ ]:
# Step 9.2 : Apply rules -> accepted / rejected masks
features["reason"] = features.apply(decide_reason, axis=1)
features["merge"]  = features["reason"].str.startswith("accept")

accepted = features[features["merge"]].copy()
rejected = features[~features["merge"]].copy()

print("accepted:", len(accepted), " rejected:", len(rejected))
print("accepted via gazetteer (same_inst):", int(accepted["same_inst"].sum()))
print("\nrule attribution:")
print(features["reason"].value_counts().to_string())
accepted.to_parquet(art("accepted_pairs.parquet"), index=False)


In [ ]:
# Step 9.3 : Validation - inspect accepted + rejected, tune thresholds
print("===== ACCEPTED sample =====")
display(accepted.sample(min(1000, len(accepted)), random_state=2)[
    ["financer1", "financer2", "cosine", "token_set", "rapidfuzz", "levenshtein", "same_inst"]])

print("===== REJECTED sample =====")
display(rejected.sample(min(1000, len(rejected)), random_state=3)[
    ["financer1", "financer2", "cosine", "token_set", "rapidfuzz", "levenshtein",
     "core_sim", "core_shared", "inst_conflict", "geo_conflict"]])

In [ ]:
# Step 9.4 : Validation - the high-cosine rejects are where false negatives hide
hi = rejected[rejected["cosine"] >= 0.95].sort_values("cosine", ascending=False)
print("high-cosine rejects:", len(hi))
display(hi.head(300)[["financer1", "financer2", "cosine", "core_sim", "core_shared",
                      "inst_conflict", "geo_conflict", "desc_conflict", "core_empty"]])

## Phase 10 : Union-Find (Disjoint Set)
Merge only approved pairs into clusters over uids, then project `cluster_id` back onto rows.

In [ ]:
# Step 10.1 : Union-Find with path compression + union by rank
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]   # path compression
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1

In [ ]:
# Step 10.2 : Constrained agglomeration - approved pairs + representative guard, then map to rows
#
# Plain Union-Find is a transitive closure: A~B and B~C merge A with C even though nothing ever
# compared them. Over 700k co-operative names that chained 24k distinct names (INDUSIND, IDFC,
# district co-op banks, credit societies) into one component.
#
# Fix: process edges best-first (gazetteer proofs, then descending cosine) and once a component
# holds >= rep_check_min_size distinct names, an incoming edge must ALSO pass decide() between
# the two component REPRESENTATIVES - not just between the two endpoint names.

# uid is assigned in Step 5.2; rebuild it if the kernel restarted (ordering matches the embeddings)
if "uid" not in df.columns:
    _uniq = pd.read_parquet(art("uniq_names.parquet"))["canonical_name"]
    _u = df["canonical_name"].map({n: k for k, n in enumerate(_uniq)})
    assert _u.notna().all(), "canonical_name missing from uniq_names.parquet - re-run 4.5 -> 5.2"
    df["uid"] = _u.astype("int64")
    print("uid rebuilt from uniq_names.parquet |", df["uid"].nunique(), "uids")

emb = np.load(art("uniq_embeddings.npy"), mmap_mode="r")   # L2-normalized -> dot == cosine
n_uid = int(emb.shape[0])
assert int(df["uid"].max()) < n_uid, "df uids exceed the embedding matrix - artifacts out of sync"
if len(accepted):
    assert int(accepted[["i", "j"]].to_numpy().max()) < n_uid, "pair uids exceed the embeddings"

# per-uid metadata, reindexed so array position == uid
_meta = (df.sort_values("row_id").drop_duplicates("uid")
           .set_index("uid")[["canonical_name", "institution_name", "anchored"]]
           .reindex(range(n_uid)))
canon = _meta["canonical_name"].fillna("").to_numpy()
inst  = _meta["institution_name"].fillna("").to_numpy()
anch  = _meta["anchored"].fillna(False).to_numpy()

_vec = {}
def _rep_vec(u):
    v = _vec.get(u)
    if v is None:
        if len(_vec) > 200_000:      # bound the cache; roots are re-read on demand
            _vec.clear()
        v = np.asarray(emb[u], dtype="float32")
        _vec[u] = v
    return v

uf   = UnionFind(n_uid)
size = np.ones(n_uid, dtype="int64")          # distinct names per component
MIN_SZ = CONFIG["cluster"]["rep_check_min_size"]

# best-first: gazetteer-proven edges, then highest cosine. A component's representative is
# therefore its strongest-evidence member, not an arbitrary one.
edges = accepted.sort_values(["same_inst", "cosine"], ascending=False)

n_merge = n_block = 0
blocked = []
for e in tqdm(edges.itertuples(index=False), total=len(edges), desc="agglomerate"):
    ra, rb = uf.find(int(e.i)), uf.find(int(e.j))
    if ra == rb:
        continue
    if e.same_inst != 1 and max(size[ra], size[rb]) >= MIN_SZ:
        cos = float(_rep_vec(ra) @ _rep_vec(rb))
        f = pair_features(canon[ra], canon[rb], cos,
                          inst[ra], inst[rb], bool(anch[ra]), bool(anch[rb]))
        why = decide_reason(f)
        if not why.startswith("accept"):
            n_block += 1
            if len(blocked) < 5000:
                blocked.append((canon[ra], canon[rb], round(cos, 4), why,
                                int(size[ra]), int(size[rb])))
            continue
    total = size[ra] + size[rb]
    uf.union(ra, rb)
    size[uf.find(ra)] = total
    n_merge += 1

blocked = pd.DataFrame(blocked, columns=["rep1", "rep2", "cos", "reason", "size1", "size2"])
uid_cluster = np.array([uf.find(x) for x in range(n_uid)])
df["cluster_id"] = uid_cluster[df["uid"].to_numpy()]

print(f"edges: {len(edges)} | merged: {n_merge} | blocked by representative guard: {n_block}")
print("clusters:", df["cluster_id"].nunique(), "| largest component (distinct names):", int(size.max()))
df[["canonical_name", "cluster_id"]].head(10)


In [ ]:
# Step 10.2b : Validation - what the representative guard refused to merge
#
# Every row here is a chain that plain Union-Find would have closed. Scan for pairs that SHOULD
# have merged: those are false negatives and mean a guard in Step 9.1 is too tight.
print("blocked edges:", len(blocked))
if len(blocked):
    print(blocked["reason"].value_counts().to_string())
    display(blocked.sort_values("cos", ascending=False).head(50))


In [ ]:
# Step 10.3 : Validation - cluster size distribution
sizes = df["cluster_id"].value_counts()
print("total clusters :", sizes.shape[0])
print("singletons     :", (sizes == 1).sum())
print("avg size       :", round(sizes.mean(), 3))
print("largest 10     :\n", sizes.head(10))

In [ ]:
# Step 10.4 : Inspect the largest clusters (over-merge check)
# Big clusters are EXPECTED here - a national bank legitimately has thousands of branch strings.
# What you are looking for is two different institutions inside one cluster.
for cid in sizes.head(15).index:
    members = df.loc[df.cluster_id == cid, "canonical_name"].unique()
    print(f"\n--- cluster {cid} ({sizes[cid]} rows, {len(members)} distinct names) ---")
    for m in members[:25]:
        print("   ", m)

## Phase 11 : Canonical Financer Name Selection
Pick one representative per cluster.

Preference order: **gazetteer institution name** (if any member anchored) -> cleanest label ->
most frequent `original_name` -> longest. Anchored clusters take the gazetteer's spelling because
it is the officially correct one, not merely the most common OCR of it.

In [ ]:
# Step 11.1 : Choose one representative financer per cluster
_HONORIFICS  = {"MR", "MRS", "MS", "M/S", "SHRI", "SHREE", "SMT", "SRI", "MISS", "DR"}
_BAD_LEADING = {"AND", "OF", "THE", "FOR"}

def _is_clean_label(name: str) -> bool:
    toks = clean_name(name).split()
    if not toks or toks[0] in _BAD_LEADING:
        return False
    return not (_HONORIFICS & set(toks))

# ---- 1. clusters that anchored to a gazetteer institution take its canonical spelling ----
anchored_rows = df[df["anchored"]]
inst_by_cluster = (anchored_rows.groupby(["cluster_id", "institution_name"], sort=False)
                                .size().rename("n").reset_index()
                                .sort_values(["cluster_id", "n"], ascending=[True, False],
                                             kind="mergesort")
                                .drop_duplicates("cluster_id")
                                .set_index("cluster_id")["institution_name"])

# ---- 2. everything else: cleanest, then most frequent, then longest original_name ----
cand = (df.groupby(["cluster_id", "original_name"], sort=False)
          .size().rename("freq").reset_index())

_ok = {n: _is_clean_label(n) for n in cand["original_name"].unique()}
cand["clean_ok"] = cand["original_name"].map(_ok).astype(int)
cand["nlen"]     = cand["original_name"].str.len()

cand = cand.sort_values(["cluster_id", "clean_ok", "freq", "nlen"],
                        ascending=[True, False, False, False], kind="mergesort")

cluster_canon = (cand.drop_duplicates("cluster_id")[["cluster_id", "original_name"]]
                     .rename(columns={"original_name": "canonical_financer"})
                     .reset_index(drop=True))

# gazetteer name wins where available
cluster_canon["institution_name"] = cluster_canon["cluster_id"].map(inst_by_cluster)
cluster_canon["canonical_financer"] = cluster_canon["institution_name"].fillna(
    cluster_canon["canonical_financer"])

cluster_canon["cluster_size"] = cluster_canon["cluster_id"].map(df["cluster_id"].value_counts())

assert cluster_canon["cluster_id"].is_unique
assert cluster_canon["cluster_size"].sum() == len(df), "cluster sizes do not cover every row"

print("clusters:", len(cluster_canon),
      "| named from gazetteer:", int(cluster_canon["institution_name"].notna().sum()))
cluster_canon.head(10)

In [ ]:
# Step 11.2 : Attach canonical financer back to every row + persist
df = df.drop(columns=["canonical_financer"], errors="ignore")   # idempotent re-run
df = df.merge(cluster_canon[["cluster_id", "canonical_financer"]], on="cluster_id", how="left")
assert df["canonical_financer"].notna().all(), "some rows got no canonical financer"

df.to_parquet(art("names_clustered.parquet"), index=False)
cluster_canon.to_parquet(art("cluster_canonical.parquet"), index=False)
cluster_canon.sort_values("cluster_size", ascending=False).head(30)

In [ ]:
# Step 11.3 : Validation - inspect the largest 100 clusters
top100 = cluster_canon.sort_values("cluster_size", ascending=False).head(100)
for row in top100.itertuples(index=False):
    members = df.loc[df.cluster_id == row.cluster_id, "original_name"].unique()[:15]
    print(f"\n[{row.cluster_id}] size={row.cluster_size} canonical={row.canonical_financer!r}")
    for m in members:
        print("   ", m)

In [ ]:
# Step 11.4 : Export the unique cluster -> canonical financer mapping
unq = df.drop_duplicates("cluster_id")[["cluster_id", "canonical_financer"]]
unq.to_csv("unique_financer_cluster_with_name_v1.csv", index=False)
print("clusters exported:", len(unq))
unq.head(20)

In [ ]:
# Step 11.5 : Ground-truth template - sample pairs for manual labelling
gt = pd.concat([
    accepted.sample(min(500, len(accepted)), random_state=21).assign(predicted="merge"),
    rejected[rejected["cosine"] >= 0.90].sample(min(500, len(rejected)), random_state=22)
            .assign(predicted="no_merge"),
])[["financer1", "financer2", "cosine", "core_sim", "core_shared", "predicted"]]
gt["is_same_financer"] = ""      # fill in by hand: 1 / 0
gt.to_csv(art("ground_truth_template.csv"), index=False)
print("labelling template written:", art("ground_truth_template.csv"), len(gt), "pairs")

## Phase 12 : Incremental Pipeline
Resolve a **new** financer string against the existing index without re-embedding everything.

clean -> typo/abbrev -> institution resolve -> canonical -> (gazetteer shortcut) -> embed ->
search existing FAISS -> features -> rules -> join existing cluster OR create new.

The gazetteer shortcut is the important part: an anchored name whose institution already exists in
`cluster_canonical.parquet` resolves with **no embedding call at all**.

In [ ]:
# Step 12.0 : Load saved artifacts (run in a FRESH kernel, after Step 0)
#
# PREREQUISITE - also execute these def-only cells:
#   1.2 clean_name | 2.1-2.4 normalize_typos | 3.1-3.2 resolve_institution
#   4.1-4.2 canonicalize | 8.1-8.2 pair_features | 9.1 decide
import torch
import faiss
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(CONFIG["model_name"],
                            device="cuda" if torch.cuda.is_available() else "cpu")
index  = faiss.read_index(art("faiss_flatip.index"))
df     = pd.read_parquet(art("names_clustered.parquet"))
uniq_names = pd.read_parquet(art("uniq_names.parquet"))["canonical_name"]
cluster_canon = pd.read_parquet(art("cluster_canonical.parquet"))

uid_meta = (df.sort_values("row_id").drop_duplicates("uid").set_index("uid")
              [["canonical_name", "institution_name", "anchored", "cluster_id"]].sort_index())

print("index:", index.ntotal, "| uids:", len(uid_meta), "| clusters:", len(cluster_canon))

In [ ]:
# Step 12.1 : Resolve a single new financer name
_inst_to_cluster = (cluster_canon.dropna(subset=["institution_name"])
                                 .set_index("institution_name")["cluster_id"].to_dict())

def resolve_new_financer(raw_name: str):
    # 1. clean -> typo/abbrev -> institution -> canonical
    c    = clean_name(raw_name)
    cf   = normalize_typos(c)
    inst, branch, anchored = resolve_institution(cf)
    cn   = canonicalize(inst)

    # 2. gazetteer shortcut - no embedding needed
    if anchored and inst in _inst_to_cluster:
        return {"canonical": cn, "institution": inst, "branch_hint": branch,
                "cluster_id": int(_inst_to_cluster[inst]), "new_cluster": False,
                "matched_via": "gazetteer"}

    # 3. embed + search existing FAISS
    vec = model.encode([cn], normalize_embeddings=CONFIG["normalize"],
                       convert_to_numpy=True).astype("float32")
    s, nbr = index.search(vec, CONFIG["top_k"])

    # 4. features + rule engine over each neighbor above the gate
    best = None
    for rank in range(nbr.shape[1]):
        j = int(nbr[0, rank]); cos = float(s[0, rank])
        if cos < CONFIG["rules"]["min_cosine_gate"]:
            break
        meta = uid_meta.loc[j]
        f = pair_features(cn, meta["canonical_name"], cos,
                          inst, meta["institution_name"], anchored, bool(meta["anchored"]))
        if decide(f) and (best is None or cos > best[1]):
            best = (int(meta["cluster_id"]), cos, meta["canonical_name"])

    if best is None:
        return {"canonical": cn, "institution": inst, "branch_hint": branch,
                "cluster_id": None, "new_cluster": True, "matched_via": None}
    return {"canonical": cn, "institution": inst, "branch_hint": branch,
            "cluster_id": best[0], "new_cluster": False,
            "matched_via": f"{best[2]!r} @ cos={best[1]:.4f}"}

In [ ]:
# Step 12.2 : Demo
for probe in [
    "STATE BANK OF INDIA, KOLLAM BRANCH",
    "CHOLAMANDALAM INVE.&FIN.COM. LTD.",
    "SHRRAM  CTY  UNION FIIN LTD",
    "SOME BRAND NEW NIDHI LIMITED",
]:
    print(probe, "->", resolve_new_financer(probe))

In [ ]:
# Step 12.3 : Persisting a new financer
# To append: encode its canonical name, index.add(vec), append a row to df with the resolved or
# newly-created cluster_id, then re-save names_clustered.parquet + the FAISS index.
# No need to regenerate every embedding.